# **Group 07**

# **Determinants of Vehicle Valuation in the Sri Lankan Online Secondary Market**
A Multi-Dimensional Analysis of the 2025 Import Liberalization Phase

# **SriLankan Vehicle(Car) Price Dataset**

*[Sri Lanka Used Car Market Dataset: Insights from Ikman.lk & Riyasewana.lk](https://www.kaggle.com/datasets/prasadnirmal/srilankan-second-vehiclecar-price-dataset)*



# **Import python libraries**


In [ ]:
%pip install kagglehub numpy pandas matplotlib seaborn statsmodels diptest prince scikit-learn scikit-posthocs


In [ ]:

# Utility / external
import kagglehub
from sklearn.model_selection import train_test_split
import os
import math
import itertools
import pandas as pd
import numpy as np


# Visualization
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
from IPython.display import display


# Statistical tests
from scipy.stats import (
    normaltest,
    levene,
    chi2_contingency,
    skew,
    kurtosis,
    kruskal,
    mannwhitneyu,
    shapiro,
    spearmanr
)


# Post-hoc tests
import scikit_posthocs as sp
import diptest  # Hartigan's dip test for unimodality


# Feature selection / scaling
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_regression


# Model matrix utilities
from patsy import dmatrices


# Clustering / mixture models
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, silhouette_samples
import prince  # FAMD (Factor Analysis of Mixed Data)


# Regression diagnostics
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
sns.set_theme()

In [ ]:
# Tree-based ensemble models for regression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Linear regression models and regularized variants
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet

# Single decision tree regression model
from sklearn.tree import DecisionTreeRegressor

# Preprocessing tools for feature scaling and categorical encoding
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Tool to apply different preprocessing steps to different column types
from sklearn.compose import ColumnTransformer

# Pipeline to combine preprocessing and modeling into a single workflow
from sklearn.pipeline import Pipeline

# Evaluation metrics to assess regression model performance
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    make_scorer
)


#KFold cross-validation
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV

# Utility for saving and loading trained models
import pickle

# To test heteroscedasticity in regression residuals
from statsmodels.stats.diagnostic import het_breuschpagan



# **Load Dataset**

In [ ]:

# Download latest version
path = kagglehub.dataset_download("prasadnirmal/srilankan-second-vehiclecar-price-dataset")



In [ ]:
df_original = pd.read_csv(os.path.join(path, "car_price_dataset.csv"), index_col=0)

# **Dataframe info**

In [ ]:
print("--- Dataframe info ---")
print(df_original.info(),"\n")

print("--- Number of rows and columns ---")
print(f"Rows: {df_original.shape[0]}, Columns: {df_original.shape[1]}")


In [ ]:
df_original.head()

# **Data preprocessing (Cleaning)**

## 1.Check duplicates and handle duplicates

In [ ]:
# Check the number of duplicate rows
num_duplicates = df_original.duplicated().sum()
print(f"Number of duplicate records: {num_duplicates}")


The dataset was examined for duplicate observations to ensure data integrity. The analysis identified 18 duplicate records. These duplicates were removed to prevent over-representation of identical observations and to maintain the validity of subsequent statistical analyses.

####Drop duplicates

In [ ]:

df_original.drop_duplicates(ignore_index=True, inplace=True)

ignore_index=True: Resets the index so that rows are renumbered sequentially (0, 1, 2, …) after duplicates are removed.

inplace=True: Applies the operation directly to the original DataFrame without creating a new object.

##2. Check missing values and handle missing values

In [ ]:
df_original.isnull().sum()

In [ ]:
num_missing = df_original.isnull().sum().sum()
print(f"Total missing values: {num_missing}")


The dataset was examined for missing values prior to analysis. As shown by the missing value summary, all variables contained zero missing observations. Therefore, no imputation or deletion procedures were required

## 3.After handling duplicates and missing values

In [ ]:
print(f"Number of duplicates: {num_duplicates}")
print(f"Number of missing values: {num_missing}")
print(f"Dataset shape after cleaning: {df_original.shape}\n")


## **4.Remove 'date' variable'**

The date variable represents the advertising or listing date of the observation. In this dataset, it does not contain predictive information about the target variable (e.g., price, sales, or outcome of interest). Including it could introduce noise rather than useful signal, as the actual date itself does not determine the outcome.

Therefore, the date column is removed during preprocessing to simplify the model, reduce dimensionality, and prevent the model from overfitting on irrelevant temporal information.

In [ ]:
# Drop the 'Date' column
df_original.drop(columns=['Date'],inplace=True)

# Verify it's gone
df_original.info()


##5.Remove "New" Condition vehicles since we only analys the used car  market.

In [ ]:
df_original = df_original[df_original['Condition'] == 'USED'].copy()
print(df_original['Condition'].value_counts())

##6.After data preprocessing

In [ ]:
print(f"Dataset shape after pre processing: {df_original.shape}\n")


In [ ]:
df_original.head()

In [ ]:
df_original.describe()

In [ ]:
df_original.describe(include='O')

In [ ]:
df_original.shape

# **Working Dataset**

df was used as the working copy. This ensures that the original dataset remains unchanged, while all operations are performed on df for convenience and consistency.

In [ ]:
df = df_original.copy()


In [ ]:
df.info()

# **Feature Engineering**

###**1.Calculate Vehicle Age**

Create a new helper column 'Age' representing the age of the vehicle
Calculated as the difference between the current year (2026) and the Year of Manufacture (YOM)


In [ ]:
#Create a new helper column for age of a vehicle using YOM variable
df['Age'] = 2026 - df['YOM']

#Age - info
df['Age'].info()

###**2.year_category**

We created a new helper column called Year_Category to classify vehicles based on their Year of Manufacture (YOM). This categorization simplifies analysis by grouping years into broader categories:

Old: Year is before 2010

Intermediate: Year is from 2010 to 2017

Modern: Year is 2018 or later

Age=2025−YOM

In [ ]:
def classify_year(year):
    if year < 2010:
        return "Old"
    elif year < 2018:
        return "Intermediate"
    else:
        return "Modern"

df['Year_Category'] = df['YOM'].apply(classify_year)

###**3.Engine Size Categories**


In [ ]:
# Displays all unique engine size values (in cc) as a list to understand the range and help with categorization
print(df["Engine (cc)"].unique().tolist())

We created a new helper column Engine_Segment to group vehicles by engine capacity (cc). The categories are:

Micro (<800cc)

Compact (800-1200cc)

Mid-Range (1200-1600cc)

Large (>1600cc)

Unknown for missing values

In [ ]:
#Engine_segment is a helper column.
def classify_engine(cc):
    # Handle missing values just in case
    if pd.isna(cc):
        return "Unknown"

    cc = float(cc)

    if cc < 800:
        return "Micro (<800cc)"
    elif cc <= 1200:
        return "Compact (800-1200cc)"
    elif cc <= 1600:
        return "Mid-Range (1200-1600cc)"
    else:
        return "Large (>1600cc)"

# Create the new column
df['Engine_Segment'] = df['Engine (cc)'].apply(classify_engine)

# Check the count of cars in each group
print(df['Engine_Segment'].value_counts())

###**4.Group brands by frequency to reduce sparsity**

Many brands appear only a few times in the dataset, which can introduce noise
and unstable estimates when used as categorical features.
 Strategy:
 - Brands with >100 listings are kept as-is (sufficient data to learn from)
 - Brands with 10–100 listings are grouped into 'OTHER'
 - Brands with <10 listings are grouped into 'RARE'

 This reduces the number of unique categories, improves model stability,
 and helps prevent overfitting while preserving information from frequent brands.






In [ ]:
# Count number of listings per brand
brand_counts = df['Brand'].value_counts()

def group_brand(x):
    count = brand_counts[x]
    if count > 100:          # Keep brands with more than 100 listings as-is
        return x
    elif count >= 10:         # 10–100 listings → OTHER
        return 'OTHER'
    else:                     # <10 listings → RARE
        return 'RARE'

# Apply to create new column
df['brand_grouped'] = df['Brand'].apply(group_brand)

# Check the result
df['brand_grouped'].value_counts()

###**5.Rename columns(original columns)**

In [ ]:
def clean_column_name(col_name):
    """
    Converts column names to snake_case.
    - Lowercases the string.
    - Replaces specific patterns like '(cc)' and '(KM)' first.
    - Replaces spaces and hyphens with underscores.
    - Ensures no double underscores.
    """
    col_name = col_name.lower()
    # Handle specific patterns first to avoid double underscores
    col_name = col_name.replace('(cc)', '_cc')
    col_name = col_name.replace('(km)', '_km') # for millage
    # Replace other characters and clean up
    col_name = col_name.replace(' ', '_')
    col_name = col_name.replace('-', '_')
    col_name = col_name.replace('__', '_') # Remove any double underscores that might have formed
    col_name = col_name.strip('_') # Remove leading/trailing underscores
    return col_name

# Apply cleaning to all columns
df.columns = [clean_column_name(col) for col in df.columns]

# Print the new column names to verify
print(df.columns)

##**6.Converted the leasing column from text to a boolean format**

In [ ]:
df['leasing'].value_counts()

df["leasing"] becomes:

True ->leased

False -> not leased

None -> unknown

In [ ]:
def parse_leasing(x):
    if pd.isna(x):
        return None
    s = str(x).lower()
    if "no" in s and "leasing" in s:
        return False
    if "lease" in s:
        return True
    return None

df["leasing"] = df["leasing"].apply(parse_leasing)

df["leasing"].value_counts()


##**7. Towns grouped into provinces**

In [ ]:
# Town → Province mapping (Sri Lanka)
town_to_province = {
    # Western Province
    "Colombo": "Western",
    "Gampaha": "Western",
    "Negombo": "Western",
    "Kalutara": "Western",
    "Panadura": "Western",
    "Moratuwa": "Western",
    "Dehiwala-Mount-Lavinia": "Western",
    "Maharagama": "Western",
    "Kotte": "Western",
    "Wattala": "Western",
    "Ja-Ela": "Western",
    "Kelaniya": "Western",
    "Kadawatha": "Western",
    "Nugegoda": "Western",
    "Piliyandala": "Western",
    "Boralesgamuwa": "Western",

    # Central Province
    "Kandy": "Central",
    "Matale": "Central",
    "Nuwara-Eliya": "Central",
    "Gampola": "Central",
    "Nawalapitiya": "Central",
    "Hatton": "Central",

    # Southern Province
    "Galle": "Southern",
    "Matara": "Southern",
    "Hambantota": "Southern",
    "Weligama": "Southern",
    "Tangalle": "Southern",
    "Hikkaduwa": "Southern",
    "Ambalangoda": "Southern",

    # Northern Province
    "Jaffna": "Northern",
    "Vavuniya": "Northern",
    "Kilinochchi": "Northern",
    "Mullaitivu": "Northern",

    # Eastern Province
    "Batticaloa": "Eastern",
    "Trincomalee": "Eastern",
    "Ampara": "Eastern",
    "Kalmunai": "Eastern",

    # North Western Province
    "Kurunegala": "North Western",
    "Puttalam": "North Western",
    "Kuliyapitiya": "North Western",
    "Chilaw": "North Western",

    # North Central Province
    "Anuradapura": "North Central",
    "Polonnaruwa": "North Central",

    # Uva Province
    "Badulla": "Uva",
    "Bandarawela": "Uva",
    "Haputale": "Uva",
    "Welimada": "Uva",

    # Sabaragamuwa Province
    "Ratnapura": "Sabaragamuwa",
    "Kegalle": "Sabaragamuwa",
    "Balangoda": "Sabaragamuwa"
}

# Create province column
df["province"] = df["town"].map(town_to_province)

# Fill towns that were not matched
df["province"] = df["province"].fillna("Other")

# Check result
df["province"].value_counts()


##**8.Classifying Towns into Urban and Non-Urban**

That is the usual logic for creating an urban vs non-urban feature:

Urban -> towns with a Municipal Council (MC) or Urban Council (UC)

Non-Urban -> towns without MC/UC (like small towns, villages, or rural areas)

In [ ]:
urban_towns = [
    "Ambalangoda","Ampara","Anuradapura","Avissawella","Badulla","Balangoda",
    "Bandarawela","Batticaloa","Beruwala","Boralesgamuwa","Chavakacheri","Chilaw",
    "Colombo","Dambulla","Dehiwala-Mount-Lavinia","Galle","Gampaha","Gampola",
    "Hambantota","Haputale","Hatton","Hikkaduwa","Horana","Ja-Ela","Jaffna",
    "Kadugannawa","Kaduwela","Kalmunai","Kalutara","Kandy","Kattankudy",
    "Katunayake","Kegalle","Kesbewa","Kolonnawa","Kotte","Kuliyapitiya",
    "Kurunegala","Maharagama","Matale","Matara","Minuwangoda","Moratuwa",
    "Nawalapitiya","Negombo","Nuwara-Eliya","Panadura","Peliyagoda","Puttalam",
    "Ratnapura","Tangalle","Trincomalee","Vavuniya","Wattala","Wattegama",
    "Weligama"
]

df["location_type"] = df["town"].apply(
    lambda x: "Urban" if str(x).strip() in urban_towns else "Non-Urban"
)

df["location_type"].value_counts()


##**9.brand_model**


A simplified Brand–Model feature was created by extracting the first token of the model name and combining it with the grouped brand variable. This reduced naming inconsistencies while preserving meaningful vehicle-type distinctions.

Extract first word from model
df["simple_model"] = df["model"].astype(str).str.split(" ").str[0]

Converts the model column to string (safety step).

Splits each model name by spaces.

Keeps only the first word.

In [ ]:
# Extract first word from model
df["simple_model"] = df["model"].astype(str).str.split(" ").str[0]

# Combine with brand_grouped to create a unique model identifier
df["brand_model"] = df["brand_grouped"] + "_" + df["simple_model"]

# Optional: check how many unique combinations now
print(df["brand_model"].nunique())



##**10.Categorical Variables and their levels**

In [ ]:
# Selects all categorical (object-type) columns and stores their names in a list
cat_list = df.select_dtypes(include=["object","category","bool"]).columns.tolist()

# Print the list of categorical columns
print(cat_list)



In [ ]:
# Loop through each categorical column
for i in cat_list:

    # Print the column name in uppercase for clarity
    print(f"--- {i.upper()} ---")

    # Display all unique values in the column
    print(f"{i.upper()}: {df[i].unique().tolist()}")

    # Print the number of unique categories in the column
    print(f"{i.upper()}: {df[i].nunique()}")

    print('\n')


**Brand**

In [ ]:
brands = df["brand"].unique().tolist()
print(brands)
print(df["brand"].nunique())
print(df["brand"].value_counts())

**brand_grouped**

In [ ]:
brands = df["brand_grouped"].unique().tolist()
print(brands)
print(df["brand_grouped"].nunique())
print(df["brand_grouped"].value_counts())

# Save Data Frame as CSV file

df is your processed training dataset.

We are saving the datasets as CSV files

In [ ]:
"""
from google.colab import files

# save and download
df.to_csv("training_dataset.csv", index=False)
test_data.to_csv("testing_dataset.csv", index = False)

"""

# **Dataset (After Feature Engineering)**

In [ ]:
df.info()

In [ ]:
df.head()

In [ ]:
#summary statistics for numerical variables
df.describe()

In [ ]:
#summary statistics for categorical variables
df.describe(include='O').style

In [ ]:
df.shape

##Categorical and Numerical Cloumns

In [ ]:
# Numeric columns
numeric_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Categorical columns
categorical_cols = df.select_dtypes(include=['object','bool']).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# **EDA(Exploratory Data Analysis)**

##**Mutual Information**

Mutual Information (MI) is a statistical measure of the dependency between two variables. It comes from information theory and tells you how much knowing one variable reduces uncertainty about the other.

In [ ]:
# Split features and target
X = df.drop(columns=['price'])
y = df['price']

# Columns to exclude from MI computation
excluded_cols = ['yom', 'engine_cc', 'date', 'model','brand_grouped','year_category','town','price_transformed','simple_model','millage_km','millage_resid']

# Encode categorical features as integers
X_encoded = X.copy()
# Drop excluded columns (safely)
X_encoded = X_encoded.drop(columns=excluded_cols, errors='ignore')

# Convert object columns to integer codes
for col in X_encoded.select_dtypes(include=['object','bool']).columns:
    X_encoded[col] = X_encoded[col].astype('category').cat.codes

# Compute Mutual Information
mi = mutual_info_regression(X_encoded, y, discrete_features='auto', random_state=42)

# Create DataFrame and sort
mi_df = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Mutual_Information': mi
}).sort_values(by='Mutual_Information', ascending=False)

print(mi_df)

# Visualize top 10 Mutual Information features
top_mi_df = mi_df.head(10)

plt.figure(figsize=(10,6))
sns.barplot(
    x='Mutual_Information',
    y='Feature',
    data=top_mi_df,
    palette='viridis'
)
plt.title("Top 10 Features by Mutual Information vs Price")
plt.xlabel("Mutual Information")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## **Investigate Response Variable(price)**

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(df['price'], bins=30, kde=True)
plt.title('Distribution of Vehicle Prices')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

Vehicle prices are highly right-skewed with many high-value outliers (luxury cars).

Using the median instead of the mean gives a more reliable measure of the typical

vehicle price within each group and avoids distortion caused by extreme values.


### **Checking Bimodality of a Target Variable**

####**1.Calculate the Bimodal Coefficient and perform Hartigan's Dip Test**

The Bimodal Coefficient is a statistic that combines skewness and kurtosis to measure whether a distribution is likely bimodal.

In [ ]:
from scipy.stats import skew, kurtosis

def calculate_bimodal_coefficient(data_series):
    """
    Calculates the bimodal coefficient for a given data series.
    A value > 5/9 (approx 0.555) suggests bimodality.
    """
    n = len(data_series)
    if n < 4: # Kurtosis requires at least 4 observations
        return np.nan

    # Calculate skewness
    data_skewness = skew(data_series)

    # Calculate Pearson's kurtosis (fisher=False)
    data_kurtosis = kurtosis(data_series, fisher=False)

    # Apply the bimodal coefficient formula
    # Formula for bimodal coefficient: (skewness^2 + 1) / (kurtosis + 3 * ((n-1)^2) / ((n-2)*(n-3)))
    numerator = data_skewness**2 + 1
    denominator_term_1 = data_kurtosis
    denominator_term_2 = (3 * ((n - 1)**2)) / ((n - 2) * (n - 3))

    bimodal_coefficient = numerator / (denominator_term_1 + denominator_term_2)

    return bimodal_coefficient

# Call the function for the 'price' column
bimodal_coeff = calculate_bimodal_coefficient(df['price'])
print(f"Bimodal Coefficient for  Price: {bimodal_coeff:.3f}")

# Check for bimodality
if bimodal_coeff > (5/9):
    print("The price distribution might be bimodal.")
else:
    print("The price distribution does not appear to be bimodal by this criterion.")

**The Dip Test is a statistical test to detect unimodality vs multimodality in a distribution.**

Null hypothesis: The distribution is unimodal.

Alternative hypothesis: The distribution is not unimodal (could be bimodal or multimodal).

In [ ]:
dip, p_value = diptest.diptest(df['price'])

print(f"Hartigan's Dip Statistic: {dip:.3f}")
print(f"P-value for Dip Test: {p_value:.3f}")

# Interpret the results (typically, p < 0.05 suggests significant departure from unimodality, i.e., bimodality or multimodality)
alpha = 0.05
if p_value < alpha:
    print(f"With a p-value of {p_value:.3f} (less than {alpha}), we reject the null hypothesis of unimodality. The price distribution likely exhibits bimodality or multimodality.")
else:
    print(f"With a p-value of {p_value:.3f} (greater than or equal to {alpha}), we fail to reject the null hypothesis of unimodality. The price distribution does not show significant evidence of bimodality by this test.")

#### **2.Summary of Bimodality Analysis**

### Bimodal Coefficient:
*   **Calculated Value**: 0.409
*   **Interpretation**: Since this value is less than 5/9 (approximately 0.555), the Bimodal Coefficient suggests that the price distribution does not exhibit bimodality according to this criterion.

### Hartigan's Dip Test:
*   **Dip Statistic**:0.014
*   **P-value**: 0.000
*   **Interpretation**: With a p-value of 0.000 (which is less than the significance level of 0.05), we reject the null hypothesis of unimodality. This test indicates that the price distribution likely exhibits bimodality or multimodality, suggesting a significant departure from a single-peaked distribution.

### Conclusion:
The two tests provide conflicting insights into the bimodality of the vehicle prices.
The Bimodal Coefficient suggests no bimodality, while Hartigan's Dip Test strongly suggests the presence of bimodality or multimodality.

This discrepancy indicates that while the distribution might not have the 'classic' bimodal shape that the coefficient typically identifies, it is statistically different from a simple unimodal distribution. This could be due to subtle multi-modal peaks or a complex distribution that the Dip Test is more sensitive to.

####**3.Gaussian Mixture Model (GMM)**

A GMM assumes your data is generated from a mixture of Gaussian distributions.

A Gaussian Mixture Model (GMM) is a probabilistic model that assumes data points are generated from a mixture of several Gaussian (normal) distributions with unknown parameters. Unlike hard clustering methods such as K-Means which assign each point to a single cluster based on the closest centroid, GMM performs soft clustering by assigning each point a probability of belonging to multiple clusters.



Each Gaussian has its own mean and variance.

By fitting a GMM, you can identify multiple peaks (modes) in the data.

In [ ]:
# Reshape the data for GMM
price_data= df['price'].values.reshape(-1, 1)

# Initialize GaussianMixture model with 2 components
gmm = GaussianMixture(n_components=2, random_state=42)

# Fit the GMM model to the data
gmm.fit(price_data)

# Print the estimated parameters
print(f"Estimated Means: {gmm.means_.flatten()}")
print(f"Estimated Standard Deviations: {np.sqrt(gmm.covariances_).flatten()}")
print(f"Estimated Mixing Proportions (Weights): {gmm.weights_}")

##### **Calculate Ashman's D**

Use the estimated means and standard deviations from the two components of the Gaussian Mixture Model to calculate Ashman's D. Ashman's D quantifies the degree of separation between two normal components.




In [ ]:
# Extract means and standard deviations from the GMM model
mu1, mu2 = gmm.means_.flatten()
sigma1, sigma2 = np.sqrt(gmm.covariances_).flatten()

# Calculate Ashman's D
ashmans_d = np.abs(mu1 - mu2) / np.sqrt((sigma1**2 + sigma2**2) / 2)

print(f"Ashman's D: {ashmans_d:.3f}")

#####**Interpret Ashman's D**
Display the calculated Ashman's D value and discuss its implications for the bimodality of the price distribution.

### Ashman's D Analysis:
*   **Calculated Value**: 1.381
*   **Interpretation**: Ashman's D quantifies the degree of separation between two normal distributions. Generally:
    *   D > 2: Excellent separation, suggesting distinct modes.
    *   1.5 < D <= 2: Good separation, indicating two identifiable modes.
    *   D <= 1.5: Poor separation, suggesting the modes are heavily overlapping or not well-defined.

    The calculated Ashman's D of 1.381 suggests a good separation between the two modes identified by the Gaussian Mixture Model for the price distribution. This indicates that while the modes are not completely distinct, they are sufficiently separated to be considered two different underlying distributions.

### Comprehensive Assessment of Bimodality:

**Previous Findings:**
*   **Bimodal Coefficient**: 0.409 . This value is less than 5/9 (approximately 0.555), suggesting that by this criterion, the  price distribution does not exhibit strong bimodality.
*   **Hartigan's Dip Test**: Dip Statistic = 0.014, P-value = 0.000. With a p-value less than 0.05, we reject the null hypothesis of unimodality, providing strong statistical evidence for bimodality or multimodality.

**Integration and Conclusion:**
The three tests provide a nuanced view of the  vehicle price distribution:

1.  **Conflicting Signals**: The Bimodal Coefficient (0.4o9) suggests a lack of strong bimodality, often requiring a more pronounced 'valley' between peaks to exceed its threshold. This might indicate that visually, the separation isn't as stark as the coefficient would prefer.

2.  **Statistical Evidence for Multi-modality**: Hartigan's Dip Test (p-value = 0.000) strongly rejects unimodality. This is a sensitive test, indicating that the distribution is statistically different from a single-peaked distribution, even if the peaks are not perfectly distinct.

3.  **Quantified Separation**: Ashman's D (1.381) corroborates the idea of distinct, though potentially overlapping, modes. A value of 1.381 falls into the 'good separation' category, reinforcing the idea that the two components identified by the GMM are indeed meaningful and not just random fluctuations.



###**Log Transformation of Price and engine_cc to Reduce Skewness**

In [ ]:
df["price_transformed"] = np.log1p(df["price"])

In [ ]:
sns.histplot(df["price_transformed"], bins=30, kde=True)
plt.title("Log-Transformed Price Distribution")
plt.xlabel("Log(Price + 1)")
plt.ylabel("Count")
plt.show()

In [ ]:
df["log_engine_cc"] = np.log1p(df["engine_cc"])

## **Distribution of Numeric variables**

In [ ]:
def plot_hist_box(df, numeric_cols, bins=30, exclude_cols=None):
    if exclude_cols is None:
        exclude_cols = []

    for col in numeric_cols:
        if col in exclude_cols:
            continue

        data = df[col].dropna()

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Histogram with KDE
        sns.histplot(data, bins=bins, kde=True, ax=axes[0],color='skyblue', alpha=0.7)
        axes[0].set_title(f"Distribution of {col}")
        axes[0].set_xlabel(col)
        axes[0].set_ylabel("Density")

        # Boxplot
        sns.boxplot(x=data, ax=axes[1], color='lightgreen')
        axes[1].set_title(f"Distribution of {col}")
        axes[1].set_xlabel(col)

        plt.tight_layout()
        plt.show()


In [ ]:
plot_hist_box(df,numeric_cols,exclude_cols=["YOM", "price"])


### **Skewness of Numerical Variables**

In [ ]:
# Select only numeric columns
numeric_cols_train = df.select_dtypes(include=np.number).columns

# Calculate skewness for each numeric column
skewness_values = df[numeric_cols_train].apply(lambda x: skew(x.dropna()))

# Create a DataFrame for better display
skewness_df = pd.DataFrame({
    'Feature': skewness_values.index,
    'Skewness': skewness_values.values
}).sort_values(by='Skewness', ascending=False)

display(skewness_df)

# Visualize skewness
plt.figure(figsize=(10, 6))
sns.barplot(x='Skewness', y='Feature', data=skewness_df, palette='viridis', hue='Feature', legend=False)
plt.title('Skewness of Numerical Features (Excluding Total_Cost_USD)')
plt.xlabel('Skewness Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

#### ***Interpretation of Skewness***:
*   **Positive Skew (Right-skewed)**: The tail on the right side of the distribution is longer or fatter than the left side. The mean is greater than the median. This often indicates the presence of outliers on the higher end.
*   **Negative Skew (Left-skewed)**: The tail on the left side of the distribution is longer or fatter than the right side. The mean is less than the median. This often indicates the presence of outliers on the lower end.
*   **Zero Skew**: The data is symmetric. The mean and median are approximately equal.

###**Outliers Identification**

In [ ]:
outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    total = df[col].notna().sum()

    outlier_summary.append([
        col, Q1, Q3, IQR, lower, upper, outliers, (outliers / total) * 100
    ])


In [ ]:
outlier_df = pd.DataFrame(
    outlier_summary,
    columns=['Feature', 'Q1', 'Q3', 'IQR', 'Lower_Bound', 'Upper_Bound',
             'Outlier_Count', 'Outlier_Percentage']
)

outlier_df.sort_values(by='Outlier_Percentage', ascending=False)


#####**Handling Outliers in the Dataset - Not remove or Handle**

Domain-Based Justification:

**Price Outliers**:

High-priced vehicles in the dataset correspond to luxury or premium cars (e.g., BMW, Mercedes).Removing these values would underestimate the effect of engine size, brand, and age on price, leading to biased inference.

**Engine Capacity (cc) Outliers**:

Vehicles with unusually high engine capacity are typically sports or high-performance cars.Excluding them would ignore an important segment of the market, resulting in a less generalizable model.

**Age / Year of Manufacture Outliers**:

Very old vintage cars or recently imported vehicles appear as outliers in age and year of manufacture.
These cars have distinct pricing patterns and contribute valuable information to understand price trends across the market.

**Mileage Outliers:**

Low mileage vehicles are often recently imported or barely used, which is common in the Sri Lankan market.

**Conclusion:**
Given that these outliers are representative of real market segments, we retain them in the analysis. Removing them would artificially reduce the variance, bias coefficient estimates, and weaken the model’s ability to generalize across different types of vehicles.

###**Correlation Analysis of Numeric Variables**

In [ ]:
num_cols = ['yom', 'engine_cc', 'millage_km', 'age', 'model_average']

In [ ]:
corr = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix – Numeric Variables')
plt.show()


##**Distribution of categorical variables**

###**1.Distribution of vehicles by fuel type**

In [ ]:
plt.figure(figsize=(6, 4))

ax = sns.countplot(data=df,x='fuel_type',order=df['fuel_type'].value_counts().index)

plt.title('Distribution of Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Number of Vehicles')

# Add count labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.show()


In [ ]:
fuel_counts = df['fuel_type'].value_counts()

plt.figure(figsize=(6, 6))

plt.pie(fuel_counts.values,labels=fuel_counts.index,autopct='%1.1f%%',startangle=90,
    labeldistance=1.2,   # move labels outward
    pctdistance=0.7      # keep percentages inside
)

plt.title('Fuel Type Proportion', y=1.08)
plt.axis('equal')
plt.show()


###**2.Distribution of vehicles by Brand**

In [ ]:
brand_order = df['brand'].value_counts().index

plt.figure(figsize=(10, 12))

ax = sns.countplot(data=df,x='brand',width=0.6,order=brand_order)

plt.title('Brand Distribution')
plt.xlabel('Number of Vehicles')
plt.ylabel('Brand')

# Add labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.xticks(rotation=55, ha='right')
plt.tight_layout()
plt.show()



Top 10 Brands Distribution

In [ ]:
top_n = 10
top_brands = df['brand'].value_counts().head(top_n).index

plt.figure(figsize=(8, 4))
ax = sns.countplot(data=df[df['brand'].isin(top_brands)],x='brand',order=top_brands)

plt.title('Top 10 Brands Distribution')
plt.xlabel('Brand')
plt.ylabel('Number of Vehicles')
plt.xticks(rotation=45, ha='right')

# Add labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.show()


###**3.Distribution of vehicles by Gear Type**

In [ ]:
gear_order= df['gear'].value_counts().index

plt.figure(figsize=(6, 4))

ax = sns.countplot(data=df,x='gear',order=gear_order)

plt.title('Distribution of Vehicles by Gear Types',y=1.1)
plt.xlabel('Gear Type')
plt.ylabel('Number of Vehicles')

# Add labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.show()


In [ ]:
gear_counts = df['gear'].value_counts()

plt.figure(figsize=(6, 6))

#pie chart
plt.pie(gear_counts.values,autopct='%1.1f%%',startangle=90)

#add legend
plt.legend(gear_counts.index,title='Gear Type',loc='center left',bbox_to_anchor=(1, 0.5))

plt.title('Distribution of Vehicles by Gear Types', y=1.08)
plt.axis('equal')

plt.show()


###**4.Distribution of vehicles by engine segments**

In [ ]:
engine_order = df['engine_segment'].value_counts().index

plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df,x='engine_segment',order=engine_order)

plt.title('Distribution of Engine Segments',y=1.1)
plt.xlabel('Engine Segment')
plt.ylabel('Number of Vehicles')


# Add labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.xticks(rotation=30, ha='right')

plt.show()


In [ ]:
engine_counts = df['engine_segment'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(engine_counts.values,autopct='%1.1f%%',startangle=90)

plt.legend(engine_counts.index,title='Engine Segment',loc='center left',bbox_to_anchor=(1, 0.5))

plt.title('Proportion of Engine Segments')
plt.axis('equal')
plt.tight_layout()
plt.show()


###**5.Distribution of Vehicle Year Categories**


In [ ]:
# Get order by frequency (descending)
year_order = df['year_category'].value_counts().index

plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df,x='year_category',order=year_order)

plt.title('Distribution of Vehicle Year Categories',y=1.1)
plt.xlabel('Year Category')
plt.ylabel('Number of Vehicles')

# Add count labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)


plt.show()

In [ ]:
year_counts = df['year_category'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(
    year_counts.values,
    autopct='%1.1f%%',
    startangle=90
)

plt.legend(
    year_counts.index,
    title='Year Category',
    loc='center left',
    bbox_to_anchor=(1, 0.5)
)

plt.title('Proportion of Vehicle Year Categories', y=1.08)
plt.axis('equal')
plt.tight_layout()
plt.show()


###**6.Distribution of Vehicles by Leasing Status**

In [ ]:
leasing_order = df['leasing'].value_counts().index

plt.figure(figsize=(5, 4))
ax = sns.countplot(data=df,x='leasing',order=leasing_order)

plt.title('Distribution of Vehicles by Leasing Status',y=1.1)
plt.xlabel('Leasing')
plt.ylabel('Number of Vehicles')

# Add count labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.show()


In [ ]:
df['leasing'].value_counts()

In [ ]:
leasing_counts = df['leasing'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(leasing_counts.values,autopct='%1.1f%%',startangle=90)

plt.legend(
    leasing_counts.index,
    title='Leasing',
    loc='center left',
    bbox_to_anchor=(1, 0.5)
)

plt.title('Distribution of Vehicles by Leasing Status', y=1.08)
plt.axis('equal')
plt.tight_layout()
plt.show()


###**7.Distribution of vehicles by model(Top 10 Vehicle Models)**

In [ ]:
top_n = 10
top_models = df['model'].value_counts().nlargest(top_n).index

plt.figure(figsize=(10, 6))
ax = sns.countplot(
    data=df[df['model'].isin(top_models)],
    y='model',
    order=top_models,
)

plt.title(f'Top {top_n} Vehicle Models (Most Common)')
plt.xlabel('Number of Vehicles')
plt.ylabel('Model')

# Add labels
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.tight_layout()
plt.show()


###**8.Distribution of Vehicles with Key Comfort & Convenience Features**

Covers air conditioning, power steering, power mirrors, and power windows

In [ ]:
import matplotlib.pyplot as plt

binary_cols = ['air_condition', 'power_steering', 'power_mirror', 'power_window']

# Create a 2x2 canvas
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.flatten()  # flatten to make iteration easier

for i, col in enumerate(binary_cols):
    counts = df[col].value_counts()

    axes[i].pie(
        counts.values,
        autopct='%1.1f%%',
        startangle=90,
        colors=['#66b3ff', '#ff9999']  # optional colors
    )
    axes[i].legend(
        counts.index,
        title=col.replace('_', ' ').title(),
        loc='center left',
        bbox_to_anchor=(1, 0.5)
    )
    axes[i].set_title(f'Proportion with {col.replace("_", " ").title()}', y=1.05)
    axes[i].axis('equal')

plt.tight_layout()
plt.show()


### **9. Distribution of Vehicle Brands**

In [ ]:
plt.figure(figsize=(10, 8))
brand_grouped_order = df['brand_grouped'].value_counts().index
ax = sns.countplot(data=df, y='brand_grouped', order=brand_grouped_order, palette='viridis')
plt.title('Distribution of Vehicle Brands (Grouped)')
plt.xlabel('Number of Vehicles')
plt.ylabel('Brand Grouped')

# Add count labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%d', padding=3)

plt.tight_layout()
plt.show()

###**10.Distribution of Vehicles (Listings) Across Urban and Non-Urban Areas**

In [ ]:
# Count listings by location type
loc_counts = df["location_type"].value_counts()

# Bar plot
plt.figure(figsize=(6,4))
bars= plt.bar(loc_counts.index, loc_counts.values,color=['orange','green'])
plt.title("Distribution of Vehicles (Listings) Across Urban and Non-Urban Areas",y=1.01)
plt.xlabel("Location Type")
plt.ylabel("Number of Vehicles (Listings)")
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )
plt.tight_layout()
plt.show()


### **11.Distribution of Vehicles (Listings) Across Provinces**

In [ ]:
# Count listings by province
prov_counts = df["province"].value_counts()

# Bar plot
plt.figure(figsize=(10,6))
bars = plt.bar(prov_counts.index, prov_counts.values, color='skyblue')
plt.title("Distribution of Vehicles (Listings) Across Provinces", y=1.01)
plt.xlabel("Province")
plt.ylabel("Number of Vehicles (Listings)")

# Add value labels on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )

plt.xticks(rotation=45)  # rotate province names if needed
plt.tight_layout()
plt.show()


##**GVIF**

In [ ]:
def calculate_gvif(X, groups):
    R = np.corrcoef(X.values, rowvar=False)
    det_full = np.linalg.det(R)

    results = []

    for feature, cols in groups.items():
        cols = [c for c in cols if c in X.columns]
        if len(cols) == 0:
            continue

        idx = [X.columns.get_loc(c) for c in cols]
        idx_rest = [i for i in range(X.shape[1]) if i not in idx]

        R_reduced = R[np.ix_(idx_rest, idx_rest)]
        det_reduced = np.linalg.det(R_reduced)

        gvif = det_full / det_reduced
        df = len(idx)
        gvif_adj = gvif ** (1 / (2 * df))

        results.append({
            "Feature": feature,
            "DF": df,
            "GVIF": gvif,
            "GVIF^(1/(2*DF))": gvif_adj
        })

    return (
        pd.DataFrame(results)
        .sort_values("GVIF^(1/(2*DF))", ascending=False)
        .reset_index(drop=True)
    )

In [ ]:
# First, redefine features to remove 'millage_km' due to perfect multicollinearity with 'age'

features_for_gvif = [
    "age",
    "engine_cc",
    "brand_grouped",
    "gear",
    "fuel_type",
    "province",
    "leasing",
    "condition",
    "air_condition",
    "power_steering",
    "power_mirror",
    "power_window"
]

X_gvif = df[features_for_gvif]


X_encoded_for_gvif = pd.get_dummies(X_gvif, drop_first=True)
X_encoded_for_gvif = X_encoded_for_gvif.astype(float)

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X_encoded_for_gvif),
    columns=X_encoded_for_gvif.columns,
    index=X_encoded_for_gvif.index
)


gvif_groups = {
    "age": ["age"],
    "engine_cc": ["engine_cc"],
    "brand_grouped": [c for c in X_scaled.columns if c.startswith("brand_grouped_")],
    "gear": [c for c in X_scaled.columns if c.startswith("gear_")],
    "fuel_type": [c for c in X_scaled.columns if c.startswith("fuel_type_")],
    "town": [c for c in X_scaled.columns if c.startswith("town_")],
    "leasing": [c for c in X_scaled.columns if c.startswith("leasing_")],
    "condition": [c for c in X_scaled.columns if c.startswith("condition_")],
    "air_condition": [c for c in X_scaled.columns if c.startswith("air_condition_")],
    "power_steering": [c for c in X_scaled.columns if c.startswith("power_steering_")],
    "power_mirror": [c for c in X_scaled.columns if c.startswith("power_mirror_")],
    "power_window": [c for c in X_scaled.columns if c.startswith("power_window_")],
}
gvif_df = calculate_gvif(X_scaled, gvif_groups)
display(gvif_df)


# **Objectives**

## **Objective1: Brand Equity and Depreciation Analysis**

###Initial OLS Regression

*   Purpose: Start with a simple Ordinary Least Squares (OLS) regression using price_transformed as the dependent variable and core predictors like age and mileage.
*   Reasoning: OLS provides a baseline understanding of the relationships, coefficient directions, and effect sizes. Even if assumptions are later violated, it helps guide feature selection, detect multicollinearity, and set the stage for more advanced or non-parametric models.

###Mileage Scaling

*   Purpose: Converts millage_km into units of 1,000 km.
*   Reasoning: Scaling improves numerical stability for models and makes coefficients easier to interpret.


In [ ]:
df["millage_scaled"] = df["millage_km"]/1000

In [ ]:
display(df[["millage_km","millage_scaled"]].head())

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
# Baseline regression: log(price) ~ age + mileage
model = smf.ols('price_transformed ~ age + millage_scaled', data=df).fit(cov_type='HC3')  # robust SE

print(model.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

residuals = model.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 6)) # 1 row, 2 columns

# Histogram
sns.histplot(residuals, kde=True, ax=axes[0])
axes[0].set_title('Histogram of Residuals')
axes[0].set_xlabel('Residuals')
axes[0].set_ylabel('Frequency')

# Residual vs Fitted
sns.scatterplot(x=model.fittedvalues, y=residuals, ax=axes[1], alpha=0.5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Fitted Values')
axes[1].set_xlabel('Fitted Values')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()

**Given the violation of key model assumptions, the estimated coefficients cannot be reliably interpreted for inferential purposes, and any associated hypothesis tests are invalid.**
***Age and Millage are perfectly correlated***

In [ ]:
df[['age', 'millage_scaled']].corr()

In [ ]:
beta_age = model.params['age']
beta_millage = model.params['millage_scaled']

print(f"Coefficient for age (beta_age): {beta_age:.4f}")
print(f"Coefficient for millage_scaled (beta_millage): {beta_millage:.4f}")

In [ ]:
depreciation_age = (math.exp(beta_age)-1)*100
depreciation_millage = (math.exp(beta_millage)-1)*100

print(f"Depreciation due to age: {depreciation_age:.05f}%")
print(f"Depreciation due to millage: {depreciation_millage:.05f}%")

###OLS with age only

In [ ]:
model_age = smf.ols('price_transformed ~ age', data=df).fit(cov_type='HC3')  # robust SE

print(model_age.summary())

###Residualization of Mileage:
  We regress mileage on age to isolate the component of mileage that is independent of age. The residuals from this regression represent the variation in mileage not explained by age, which we then use as a predictor in the OLS model. This approach reduces multicollinearity and allows us to estimate the unique contribution of mileage to price.

In [ ]:
import statsmodels.formula.api as smf

# Residualize mileage
resid_model = smf.ols('millage_scaled ~ age', data=df).fit()
df['mileage_resid'] = resid_model.resid

In [ ]:
df['mileage_resid'].head(10)

**Price with age and millage residuals**

In [ ]:
model_resid = smf.ols(
    'price_transformed ~ age + mileage_resid',
    data=df
).fit(cov_type='HC3')  # robust SE
print(model_resid.summary())


***Due to near-perfect correlation between age and mileage, age is used as the primary proxy for depreciation.***

In [ ]:
model_dep = smf.ols(
    'price_transformed ~ age',
    data=df
).fit(cov_type='HC3')

print(model_dep.summary())

###Baseline Brand Model (OLS):

We estimate the effect of brand_grouped on log-transformed price while controlling for age. Using OLS with robust standard errors (HC3) allows us to account for heteroscedasticity.

Reference brand: Toyota. Coefficients of other brands indicate their baseline price difference relative to Toyota.

Purpose: To quantify brand premiums and understand how different brands are associated with price after adjusting for vehicle age.

In [ ]:
model_brand = smf.ols(
    'price_transformed ~ age + C(brand_grouped, Treatment(reference="TOYOTA"))',
    data=df
).fit(cov_type='HC3')

print(model_brand.summary())



In [ ]:
r_squared_data = {
    'Model with Age Only': model_dep.rsquared,
    'Model with Age and Brand Grouped': model_brand.rsquared
}

r_squared_df = pd.DataFrame(r_squared_data.items(), columns=['Model', 'R-squared Value'])
print(r_squared_df)

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Extract coefficients & CIs
# -----------------------------
params = model_brand.params
conf = model_brand.conf_int()

# -----------------------------
# Keep only brand terms
# -----------------------------
brand_mask = params.index.str.contains("brand_grouped")

brand_premiums = pd.DataFrame({
    "term": params.index[brand_mask],
    "log_coef": params[brand_mask].values,
    "ci_lower": conf.loc[brand_mask, 0].values,
    "ci_upper": conf.loc[brand_mask, 1].values
})

# -----------------------------
# Extract clean brand names
# -----------------------------
brand_premiums["brand"] = (
    brand_premiums["term"]
    .str.extract(r"\[T\.(.+)\]")  # <-- THIS is the fix
)

brand_premiums = brand_premiums.drop(columns="term")

# -----------------------------
# Convert log coefficients to %
# -----------------------------
brand_premiums["price_premium_%"] = (np.exp(brand_premiums["log_coef"]) - 1) * 100
brand_premiums["ci_lower_%"] = (np.exp(brand_premiums["ci_lower"]) - 1) * 100
brand_premiums["ci_upper_%"] = (np.exp(brand_premiums["ci_upper"]) - 1) * 100

# -----------------------------
# Sort by premium
# -----------------------------
brand_premiums = brand_premiums.sort_values(
    by="price_premium_%",
    ascending=False
).reset_index(drop=True)

brand_premiums = brand_premiums[
    ["brand", "log_coef", "ci_lower", "ci_upper",
     "price_premium_%", "ci_lower_%", "ci_upper_%"]
]

brand_premiums


In [ ]:
df.info()

In [ ]:
sns.scatterplot(
    data=df,
    x="log_engine_cc",
    y="price_transformed"
)

###Engine-Adjusted Brand Model (OLS):
We extend the baseline brand model by including log_engine_cc to account for engine size differences across vehicles.

Purpose: To isolate the effects of brand on price while controlling for both age and engine capacity.

Interpretation:

C(brand_grouped) coefficients capture baseline price differences between brands after adjusting for age and engine size.

log_engine_cc coefficient quantifies the (multiplicative) impact of engine capacity on price.

Methodology: OLS is used with robust HC3 standard errors to account for heteroscedasticity.

This model helps us understand how brand and engine size jointly influence price, without assuming normality of residuals.

In [ ]:
model_engine = smf.ols(
    'price_transformed ~ age + C(brand_grouped, Treatment(reference="TOYOTA")) + log_engine_cc',
    data=df
).fit(cov_type='HC3')

print(model_engine.summary())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, normal_ad

# --- 1. Extract residuals and fitted values ---
residuals = model_engine.resid
fitted = model_engine.fittedvalues

# Create a single figure with a 2x2 subplot grid
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Diagnostic Plots for Engine-Adjusted Brand Model', fontsize=16)

# --- 2. Residuals vs Fitted: Check for non-linearity and heteroscedasticity ---
sns.scatterplot(x=fitted, y=residuals, alpha=0.5, ax=axes[0, 0])
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('1. Residuals vs Fitted Values')

# --- 3. Q-Q plot: Check normality of residuals ---
sm.qqplot(residuals, line='45', fit=True, ax=axes[0, 1])
axes[0, 1].set_title('2. Q-Q Plot of Residuals')

# --- 4. Histogram of residuals ---
sns.histplot(residuals, kde=True, ax=axes[1, 0])
axes[1, 0].set_xlabel('Residuals')
axes[1, 0].set_title('3. Residual Distribution')

# --- 5. Optional: Leverage and influence (Cook's distance) ---
influence = model_engine.get_influence()
cooks_d = influence.cooks_distance[0]

sns.scatterplot(x=range(len(cooks_d)), y=cooks_d, alpha=0.5, ax=axes[1, 1])
axes[1, 1].axhline(4/len(cooks_d), color='red', linestyle='--', label='Threshold 4/n')
axes[1, 1].set_xlabel('Observation')
axes[1, 1].set_ylabel("Cook's Distance")
axes[1, 1].set_title("4. Influential Points")
axes[1, 1].legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to prevent title overlap
plt.show()

# --- 6. Breusch-Pagan test for heteroscedasticity ---
# Hypotheses:
# H0: Residuals have constant variance (homoscedastic)
# H1: Residuals have non-constant variance (heteroscedastic)

bp_test = het_breuschpagan(residuals, model_engine.model.exog)
bp_labels = ['LM Statistic', 'p-value', 'F-Statistic', 'F p-value']
print("Breusch-Pagan test for heteroscedasticity:")
print(dict(zip(bp_labels, bp_test)))

# --- 7. Anderson-Darling test for normality ---
# Hypotheses:
# H0: Residuals are normally distributed
# H1: Residuals are not normally distributed

ad_stat, ad_p = normal_ad(residuals)
print(f"\nAnderson-Darling test for normality:\nStatistic = {ad_stat:.3f}, p-value = {ad_p:.3e}")

In [ ]:
import numpy as np
import pandas as pd

# Exogenous matrix and names
X = model_engine.model.exog
col_names = model_engine.model.exog_names

# Identify brand_grouped dummies
brand_cols = [name for name in col_names if "brand_grouped" in name]

groups = {
    "age": ["age"],
    "log_engine_cc": ["log_engine_cc"],
    "brand_grouped": brand_cols
}

def compute_gvif(X, groups, col_names):
    R = np.corrcoef(X, rowvar=False)
    detR = np.linalg.det(R)

    col_idx = {name: i for i, name in enumerate(col_names)}

    results = []

    for var, cols in groups.items():
        idx = [col_idx[c] for c in cols]
        R_jj = R[np.ix_(idx, idx)]
        detR_jj = np.linalg.det(R_jj)
        df = len(cols)
        gvif = detR_jj / detR
        results.append({
            "variable": var,
            "GVIF": gvif,
            "df": df,
            "GVIF^(1/(2*df))": gvif ** (1 / (2*df))
        })

    return pd.DataFrame(results)

gvif_df = compute_gvif(X, groups, col_names)
gvif_df



**Since the model assumptions are violated, we can't use this for hypothesis testing**

###Kruskal–Wallis Test for Brand Price Differences

To examine whether car prices differ systematically across brands without relying on parametric assumptions, we apply the Kruskal–Wallis H test, a non-parametric alternative to one-way ANOVA.

This test compares the distributions of log-transformed prices across multiple brand groups using rank information, making it suitable when normality and homoscedasticity assumptions are violated.

Hypotheses:

*   H₀ (Null): The distribution of price_transformed is the same across all car brands.
*   H₁ (Alternative): At least one brand has a different price distribution.

A small p-value indicates that prices are not equal across all brands, justifying further post-hoc pairwise comparisons to identify which specific brands differ.

In [ ]:
from scipy.stats import kruskal

groups = [df.loc[df['brand_grouped'] == b, 'price_transformed'] for b in df['brand_grouped'].unique()]
stat, p = kruskal(*groups)
print(f"Kruskal-Wallis H statistic: {stat:.2f}, p-value: {p:.5f}")


**We reject the null hypothesis**

###Pairwise Brand Comparisons vs Toyota (Mann–Whitney U Test)

After establishing that prices differ across brands using the Kruskal–Wallis test, we conduct pairwise non-parametric comparisons between Toyota (reference brand) and each other brand using the Mann–Whitney U test.

Hypotheses (for each brand
𝑏
b vs Toyota):

*   H₀ (Null): The distribution of price_transformed for brand
𝑏
b is the same as Toyota.
*   H₁ (Alternative): The distribution of price_transformed for brand
𝑏
b differs from Toyota

Because multiple pairwise tests are performed, Bonferroni correction is applied to control the family-wise error rate and reduce false positives.

Interpretation:

p_value_corrected < 0.05 → statistically significant price difference relative to Toyota

significant = True → reject the null hypothesis for that brand

In [ ]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

brands = [b for b in df['brand_grouped'].unique() if b != 'TOYOTA']
pvals = []

for b in brands:
    stat, p = mannwhitneyu(df.loc[df['brand_grouped']=='TOYOTA','price_transformed'],
                           df.loc[df['brand_grouped']==b,'price_transformed'],
                           alternative='two-sided')
    pvals.append(p)

# Adjust for multiple comparisons
reject, pvals_corrected, _, _ = multipletests(pvals, method='bonferroni')

result_table = pd.DataFrame({
    "brand": brands,
    "p_value_raw": pvals,
    "p_value_corrected": pvals_corrected,
    "significant": reject
})
result_table


### One-sided Pairwise Brand Comparisons (Toyota vs Selected Brands)

To examine directional price differences between Toyota and selected brands (Honda, Nissan, Mitsubishi, Tata, Mercedes-Benz), we apply the **one-sided Mann–Whitney U test** on `price_transformed`.

This **non-parametric test** compares the rank distributions of prices between two independent brands **without assuming normality**.  
A one-sided alternative is used when the research question is directional (i.e., whether one brand is priced higher than another).

**Hypotheses (for each comparison):**

- **Null hypothesis (H₀):** Toyota prices are **not lower** than the comparison brand.  
- **Alternative hypothesis (H₁):** Toyota prices are **lower** than the comparison brand.

Formally:

\[
H_0: P(\text{Toyota} \ge \text{Brand}) = 0.5 \\
H_1: P(\text{Toyota} < \text{Brand}) > 0.5
\]

**Interpretation:**

- `p-value < 0.05` → Evidence that Toyota is significantly cheaper than the comparison brand  
- `p-value ≥ 0.05` → Insufficient evidence to conclude a directional price difference

These tests provide **distribution-level, assumption-robust confirmation of relative brand pricing** and complement both the regression and omnibus non-parametric results.


In [ ]:
from scipy.stats import mannwhitneyu

toyota_prices = df.loc[df['brand_grouped']=='TOYOTA', 'price_transformed']
honda_prices = df.loc[df['brand_grouped']=='HONDA', 'price_transformed']

# one-sided test (if you want to test if Toyota > Honda)
stat, p = mannwhitneyu(toyota_prices, honda_prices, alternative='less')
print(f"One-sided Mann-Whitney U (Toyota < Honda) p-value: {p:.5f}")


In [ ]:
from scipy.stats import mannwhitneyu

toyota_prices = df.loc[df['brand_grouped']=='TOYOTA', 'price_transformed']
NISSAN_prices = df.loc[df['brand_grouped']=='NISSAN', 'price_transformed']

# one-sided test (if you want to test if Toyota > NISSAN)
stat, p = mannwhitneyu(toyota_prices, NISSAN_prices, alternative='greater')
print(f"One-sided Mann-Whitney U (Toyota > NISSAN) p-value: {p:.5f}")


In [ ]:
from scipy.stats import mannwhitneyu

toyota_prices = df.loc[df['brand_grouped']=='TOYOTA', 'price_transformed']
SUZUKI_prices = df.loc[df['brand_grouped']=='SUZUKI', 'price_transformed']

# one-sided test (if you want to test if Toyota > SUZUKI)
stat, p = mannwhitneyu(toyota_prices, SUZUKI_prices, alternative='greater')
print(f"One-sided Mann-Whitney U (Toyota > SUZUKI) p-value: {p:.5f}")


In [ ]:
from scipy.stats import mannwhitneyu

toyota_prices = df.loc[df['brand_grouped']=='TOYOTA', 'price_transformed']
MITSUBISHI_prices = df.loc[df['brand_grouped']=='MITSUBISHI', 'price_transformed']

# one-sided test (if you want to test if Toyota > MITSUBISHI)
stat, p = mannwhitneyu(toyota_prices, MITSUBISHI_prices, alternative='greater')
print(f"One-sided Mann-Whitney U (Toyota > MITSUBISHI) p-value: {p:.5f}")

In [ ]:
from scipy.stats import mannwhitneyu

toyota_prices = df.loc[df['brand_grouped']=='TOYOTA', 'price_transformed']
TATA_prices = df.loc[df['brand_grouped']=='TATA', 'price_transformed']

# one-sided test (if you want to test if Toyota > TATA)
stat, p = mannwhitneyu(toyota_prices, TATA_prices, alternative='greater')
print(f"One-sided Mann-Whitney U (Toyota > TATA) p-value: {p:.5f}")

In [ ]:
from scipy.stats import mannwhitneyu

toyota_prices = df.loc[df['brand_grouped']=='TOYOTA', 'price_transformed']
MERCEDES_BENZ_prices = df.loc[df['brand_grouped']=='MERCEDES-BENZ', 'price_transformed']

# one-sided test (if you want to test if Toyota > MERCEDES-BENZ)
stat, p = mannwhitneyu(toyota_prices, MERCEDES_BENZ_prices, alternative='less')
print(f"One-sided Mann-Whitney U (Toyota < MERCEDES-BENZ) p-value: {p:.5f}")

###Spearman Correlation: Age vs Price

We use Spearman’s rank correlation to test the association between vehicle age and log-price, since the relationship may be non-linear and OLS assumptions are not fully satisfied.

Hypotheses:

H₀: No monotonic relationship between age and price (ρ = 0)

H₁: A monotonic relationship exists (ρ ≠ 0)

A negative and significant coefficient indicates that vehicle prices decrease as age increases, confirming depreciation effects.

In [ ]:
from scipy.stats import spearmanr

rho_age, p_age = spearmanr(df['age'], df['price_transformed'])
print(f"Spearman's rank correlation coefficient (rho): {rho_age:.4f}")
print(f"P-value: {p_age:.5f}")

###Spearman Correlation: Engine Capacity vs Price

We apply Spearman’s rank correlation to assess the monotonic relationship between engine capacity (log-scaled) and log-price, without assuming linearity or normality.

Hypotheses:

H₀: No monotonic relationship between engine capacity and price (ρ = 0)

H₁: A monotonic relationship exists (ρ ≠ 0)

A positive and significant coefficient implies that vehicles with larger engines tend to have higher prices.

In [ ]:
from scipy.stats import spearmanr

rho_cc, p_cc = spearmanr(df['log_engine_cc'], df['price_transformed'])
print(f"Spearman's rank correlation coefficient (rho): {rho_cc:.4f}")
print(f"P-value: {p_cc:.5f}")


In [ ]:
pip install pingouin

###Partial Spearman Correlation: Engine Capacity and Price

**We want to assess the association between engine size (log_engine_cc) and used car listing price (price_transformed), while controlling for confounding effects of age and brand.**
Method: Partial Spearman correlation

*   Spearman correlation: Measures monotonic relationships between variables; does not assume linearity or normality.
*   Partial correlation: Measures the association between two variables after removing the influence of one or more covariates.


Covariates included:

age → older cars are generally cheaper.

brand_grouped → different brands have different baseline prices.

Hypotheses tested:

*   **Null (H₀): There is no monotonic association between engine size and price after controlling for age and brand.**
*   **Alternative (H₁): There is a monotonic association between engine size and price after controlling for age and brand.**

Interpretation of output:

r → Partial Spearman correlation coefficient: magnitude and direction of association.

p-val → Statistical significance: low p-value (< 0.05) indicates rejection of H₀.

CI95% → 95% confidence interval for the correlation estimate.


In [ ]:
# One-hot encode brand_grouped
df_encoded = pd.get_dummies(df[['price_transformed','log_engine_cc','age','brand_grouped']], drop_first=True)

import pingouin as pg

# All columns are now numeric, specify covariates
covariates = [c for c in df_encoded.columns if c not in ['price_transformed','log_engine_cc']]

partial_corr = pg.partial_corr(data=df_encoded, x='log_engine_cc', y='price_transformed', covar=covariates, method='spearman')
print(partial_corr)

In [ ]:
# One-hot encode brand_grouped
df_encoded = pd.get_dummies(df[['price_transformed','log_engine_cc','age','brand_grouped']], drop_first=True)

import pingouin as pg

# All columns are now numeric, specify covariates
covariates = [c for c in df_encoded.columns if c not in ['price_transformed','age']]

partial_corr = pg.partial_corr(data=df_encoded, x='age', y='price_transformed', covar=covariates, method='spearman')
print(partial_corr)

In [ ]:
!pip install pygam

###Generalized Additive Model (GAM)

We fit a Generalized Additive Model (GAM) to model vehicle prices while allowing for non-linear effects in continuous predictors and categorical brand differences, without imposing a strict linear structure.

Model structure:

s(log_engine_cc): **captures a potentially non-linear relationship between engine capacity and log-price**

s(age): **captures non-linear depreciation patterns over vehicle age**

f(brand): **models brand-specific baseline effects after controlling for age and engine size**

***Why GAM?***

Relaxes linearity assumptions violated in OLS

Separates smooth continuous effects from discrete brand effects

Improves interpretability compared to black-box models

This model allows us to visualize marginal effects, detect non-linear sensitivities, and isolate brand premiums in a flexible yet interpretable framework.

In [ ]:
from pygam import LinearGAM, s, f, l

# Make Toyota the reference category
df['brand_grouped'] = pd.Categorical(
    df['brand_grouped'],
    categories=['TOYOTA'] + [b for b in df['brand_grouped'].unique() if b != 'TOYOTA'],
    ordered=True
)

# Encode as numeric codes
df['brand_code'] = df['brand_grouped'].cat.codes

# Features
X = df[['log_engine_cc', 'age', 'brand_code']].values
y = np.log(df['price'].values)

# Fit GAM with Toyota as reference
gam = LinearGAM(
    s(0) +           # smooth effect of log(engine_cc)
    f(2) +           # brand premium (Toyota is baseline)
    s(1, by=2)       # brand-specific age depreciation
).fit(X, y)

# Check summary
gam.summary()

### Generalized Additive Model (GAM): Specification, Diagnostics

This document describes the estimation, diagnostics, and interpretation of a Generalized Additive Model (GAM) used to analyze vehicle prices. The response variable is log-transformed to stabilize variance and allow multiplicative interpretation.

---

#### Model specification

We fit a GAM where log-transformed price is modeled as a smooth, potentially non-linear function of engine capacity and vehicle age, while controlling for brand effects. Smoothing parameters are estimated using REML for stable inference.

```r
gam_model_v2 <- gam(
  price_transformed ~
    s(log_engine_cc, k = 20) +
    s(age, k = 20) +
    brand_grouped,
  data = df,
  method = "REML"
)
```

---

#### Model diagnostics

##### Basis adequacy and residual diagnostics

The following diagnostic checks ensure that the basis dimensions are sufficiently large and that residual patterns do not indicate model misspecification.

```r
gam.check(gam_model_v2)
```

---

#### Visualization of smooth terms

The smooth effects of log(engine capacity) and age are visualized below. Partial residuals are overlaid to assess the adequacy of the fitted smooths.

```r
plot(
  gam_model_v2,
  pages = 1,
  residuals = TRUE,
  pch = 1,
  cex = 0.5,
  scheme = 1
)
```

The diagnostic check is repeated after visual inspection to confirm that the wiggliness limits were appropriate.

```r
gam.check(gam_model_v2)
```

---

#### Interpreting smooth effects on the original price scale

Because the response is log-transformed, the inverse transformation is defined to express effects on the original price scale.

```r
unlog_price <- function(x) {
  exp(x)
}
```

The smooth terms are plotted on the real price scale by shifting the curves by the intercept and applying the inverse log transformation.

```r
plot(
  gam_model_v2,
  pages = 1,
  scheme = 1,
  shade = TRUE,
  shade.col = "lightblue",
  shift = coef(gam_model_v2)[1],
  trans = unlog_price,
  seWithMean = TRUE,
  main = "Effect of Engine Capacity and Age on Price"
)
```

---

#### Concurvity assessment

Concurvity is examined to assess non-linear dependence between predictors, which is the GAM analogue of multicollinearity.

Overall concurvity:

```r
concurvity(gam_model_v2, full = TRUE)
```

Pairwise concurvity estimates:

```r
cc <- concurvity(gam_model_v2, full = FALSE)
cc$estimate
```

---

#### Model summary

```r
summary(gam_model_v2)
```

The summary reports parametric coefficients, smooth term significance, estimated degrees of freedom, and overall model fit statistics.

---

#### Interpretation of parametric effects

For parametric (linear) terms, log-scale coefficients are converted into percentage changes for interpretability.

```r
coeffs <- summary(gam_model_v2)$p.table
estimates <- coeffs[, "Estimate"]
percent_impact <- (exp(estimates) - 1) * 100

results_table <- data.frame(
  Variable = names(estimates),
  Log_Coefficient = round(estimates, 3),
  Percent_Change = round(percent_impact, 1)
)

print(results_table)
```

The percentage change values represent the expected percentage change in price associated with each parametric effect.


In [ ]:
import matplotlib.pyplot as plt

# Get unique brands (assuming you have a list of names matching the codes)
brands = df['brand_grouped'].unique()
# OR manually list them if you want specific order
# brands = ['TOYOTA', 'MERCEDES', 'TATA']

plt.figure(figsize=(10, 6))

# Loop through each brand code to predict its specific curve
for brand_name in brands:
    brand_code = df[df['brand_grouped'] == brand_name]['brand_code'].iloc[0]

    # Create fake data for prediction:
    # Engine = mean (constant), Age = 0 to 20, Brand = current brand
    X_pred = np.zeros((100, 3))
    X_pred[:, 0] = df['log_engine_cc'].mean() # Hold engine constant
    X_pred[:, 1] = np.linspace(0, 20, 100)    # Vary Age
    X_pred[:, 2] = brand_code                 # Hold Brand constant

    # Predict
    y_pred = gam.predict(X_pred)

    # Plot
    plt.plot(np.linspace(0, 20, 100), y_pred, label=brand_name)

plt.xlabel("Age (Years)")
plt.ylabel("Predicted Log Price")
plt.title("Depreciation Profiles by Brand")
plt.legend()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

y_pred = gam.predict(X)
residuals = y - y_pred

plt.figure(figsize=(8,5))
plt.scatter(y_pred, residuals, alpha=0.5)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Predicted log(price)")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.show()


In [ ]:
import scipy.stats as stats

plt.figure(figsize=(6,6))
stats.probplot(residuals, dist="norm", plot=plt)
plt.title("QQ Plot of Residuals")
plt.show()


In [ ]:
# Engine size effect
XX = gam.generate_X_grid(term=0)
pdep, confi = gam.partial_dependence(term=0, X=XX, width=0.95)

plt.figure(figsize=(8,5))
plt.plot(XX[:,0], pdep)
plt.fill_between(XX[:,0], confi[:,0], confi[:,1], alpha=0.25)
plt.axhline(0, color='black', linestyle='--')
plt.title("Engine Size Effect")
plt.show()


In [ ]:
from pygam import LinearGAM
from sklearn.model_selection import cross_val_score
import numpy as np

gam = LinearGAM(s(0) + f(2) + s(1, by=2))

scores = cross_val_score(gam, X, y, cv=5, scoring='r2')
print("CV R²:", scores, "Mean:", np.mean(scores))

# Re-fit the gam object after cross-validation so it can be used by subsequent cells
gam.fit(X, y)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Generate Data (This comes out in Log Scale)
XX = gam.generate_X_grid(term=0)
pdep, confi = gam.partial_dependence(term=0, X=XX, width=0.95)

# DIAGNOSTIC: This should now print small numbers like "6.5 to 8.2"
print(f"X-axis (Log) range: {XX[:, 0].min():.2f} to {XX[:, 0].max():.2f}")

plt.figure(figsize=(10, 6))

# 2. Plot the Log Data directly
# The X-axis values are 6.0, 7.0, etc.
plt.plot(XX[:, 0], pdep, label="Engine Effect", color='blue', linewidth=2)
plt.fill_between(XX[:, 0], confi[:, 0], confi[:, 1], color='blue', alpha=0.15)
plt.axhline(0, color="black", linestyle="--", linewidth=1)

# 3. THE MAGIC: Custom Ticks
# We define the real CCs we want to see on the axis
real_cc_ticks = [660, 800, 1000, 1200, 1500, 1800, 2000, 2500, 3000, 4000]

# We convert them to Log Scale so matplotlib knows where to put them on the x-axis
log_tick_locs = np.log(real_cc_ticks)

# We define the limits to match your data (so the plot doesn't look empty)
# This prevents showing ticks for 4000cc if your data stops at 2000cc
data_min = XX[:, 0].min()
data_max = XX[:, 0].max()
valid_ticks = [ (loc, label) for loc, label in zip(log_tick_locs, real_cc_ticks)
                if loc >= data_min and loc <= data_max ]

if not valid_ticks: # Fallback if ranges don't match
    print("Warning: Standard ticks are outside data range. Using default.")
else:
    final_locs, final_labels = zip(*valid_ticks)
    plt.xticks(final_locs, final_labels, rotation=45)

# 4. Final Labels
plt.xlabel("Engine Capacity (CC)") # We label it as CC, even though the math is Log
plt.ylabel("Effect on Price (Log Scale)")
plt.title("Effect of Engine Size on Price (Log-Transformed Model)")
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
brands = df['brand_code'].unique()
age_grid = np.linspace(df['age'].min(), df['age'].max(), 100)

brand_cat = df['brand_grouped'].astype('category')
brand_map = dict(enumerate(brand_cat.cat.categories))

plt.figure(figsize=(10, 6))

for b in brands:
    XX = np.zeros((len(age_grid), X.shape[1]))
    XX[:, 1] = age_grid              # age
    XX[:, 2] = b                     # brand code
    XX[:, 0] = df['log_engine_cc'].mean()

    pdep = gam.partial_dependence(term=2, X=XX)

    plt.plot(age_grid, pdep, label=brand_map[b])

plt.axhline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Age (years)")
plt.ylabel("Effect on log(price)")
plt.title("Brand-Specific Depreciation Curves")
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Define the specific brands you want to plot
target_brand_names = [
    "TOYOTA", "HONDA", "NISSAN", "MITSUBISHI",
    "TATA", "MERCEDES-BENZ", "MAZDA"
]

# 2. Get the codes corresponding to these names
# We filter the dataframe to find the matching codes
selected_brands_df = (
    df[['brand_code', 'brand_grouped']]
    .drop_duplicates()
    .query("brand_grouped in @target_brand_names")
)

# 3. Setup Grid
age_grid = np.linspace(df['age'].min(), df['age'].max(), 100)

plt.figure(figsize=(10, 6))

# 4. Loop only through the SELECTED brands
for _, row in selected_brands_df.iterrows():
    b_code = row['brand_code']
    b_name = row['brand_grouped']

    # Prepare the Input Matrix
    XX = np.zeros((len(age_grid), X.shape[1]))
    XX[:, 0] = df['log_engine_cc'].mean() # Hold Engine Constant
    XX[:, 1] = age_grid                   # Vary Age
    XX[:, 2] = b_code                     # Hold specific Brand Code

    # Predict (Base Price + Depreciation + Engine Effect)
    pred = gam.predict(XX)

    # Plot
    plt.plot(age_grid, pred, label=b_name, linewidth=2)

# 5. Styling
plt.xlabel("Age (Years)")
plt.ylabel("Predicted Log Price")
plt.title("Depreciation Curves: Selected Brands")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

brand_effects = gam.coef_[gam.terms.get_coef_indices(1)]
brand_labels = (
    df['brand_grouped']
    .astype('category')
    .cat.categories
)

premium_df = pd.DataFrame({
    "brand": brand_labels,
    "log_premium": brand_effects
})

premium_df["percent_premium"] = np.exp(premium_df["log_premium"]) - 1

premium_df.sort_values("percent_premium", inplace=True)

plt.figure(figsize=(8, 6))
plt.barh(
    premium_df["brand"],
    premium_df["percent_premium"]
)
plt.axvline(0, color="black", linestyle="--", linewidth=1)
plt.xlabel("Price Premium (%)")
plt.title("Brand Premium Controlling for Age and Engine Size")
plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Setup Data & Helper Function
# Get the exact code for Toyota to use as the baseline
toyota_code = (
    df[['brand_grouped', 'brand_code']]
    .drop_duplicates()
    .query("brand_grouped == 'TOYOTA'")
    ['brand_code']
    .iloc[0]
)

# Create a grid of ages from 0 to the max age in your data
age_grid = np.linspace(df['age'].min(), df['age'].max(), 100)
brand_codes = df['brand_code'].unique()

# Define the function to predict price trajectory (Intercept + Curve)
def get_brand_price_trajectory(brand_code):
    # Create a synthetic dataset
    # Shape must match your training X: [log_engine_cc, age, brand_code, ...]
    # Assuming X has 3 columns based on your previous messages
    XX = np.zeros((len(age_grid), X.shape[1]))

    XX[:, 0] = df['log_engine_cc'].mean() # Hold Engine Constant
    XX[:, 1] = age_grid                   # Vary Age
    XX[:, 2] = brand_code                 # Specific Brand

    # Use .predict() to capture the total effect (Base Price + Depreciation)
    return gam.predict(XX)

# 2. Calculate the "Difference vs Toyota"
toyota_curve = get_brand_price_trajectory(toyota_code)

diff_df = []

for b in brand_codes:
    curve_b = get_brand_price_trajectory(b)

    # Calculate the gap (Positive = More expensive than Toyota)
    diff = curve_b - toyota_curve

    diff_df.append(pd.DataFrame({
        "age": age_grid,
        "log_diff_vs_toyota": diff,
        "brand_code": b
    }))

diff_df = pd.concat(diff_df)

# 3. Filter for specific brands to keep the plot readable
selected_brand_names = ["TOYOTA", "HONDA", "NISSAN", "MITSUBISHI", "SUZUKI", "MERCEDES-BENZ", "TATA","MAZDA"]

selected_brand_codes = (
    df.loc[df["brand_grouped"].isin(selected_brand_names),
           ["brand_grouped", "brand_code"]]
      .drop_duplicates()
)

# Merge names back onto the difference data
plot_df = diff_df.merge(
    selected_brand_codes,
    on="brand_code",
    how="inner"
)

# 4. Generate the Plot
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=plot_df,
    x="age",
    y="log_diff_vs_toyota",
    hue="brand_grouped",
    linewidth=2.5,
    palette="tab10" # Distinct colors
)

# Add reference line at 0 (Toyota Baseline)
plt.axhline(0, color="black", linestyle="--", linewidth=1.5, label="TOYOTA Baseline")

plt.xlabel("Vehicle Age (Years)", fontsize=12)
plt.ylabel("Log Price Difference vs Toyota", fontsize=12)
plt.title("Price Premium/Discount vs. Toyota Over Time", fontsize=14)
plt.legend(title="Brand", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.show()

In [ ]:
# Create the Absolute Price Plot (Real Depreciaton)
plt.figure(figsize=(10, 6))

# Loop through brands to plot their ACTUAL predicted path
for b_code, b_name in zip(plot_df['brand_code'].unique(), plot_df['brand_grouped'].unique()):

    # 1. Generate the prediction input
    XX = np.zeros((len(age_grid), X.shape[1]))
    XX[:, 0] = df['log_engine_cc'].mean() # Constant Engine
    XX[:, 1] = age_grid                   # Age 0 to Max
    XX[:, 2] = b_code                     # Current Brand

    # 2. Predict Absolute Log Price
    pred_log_price = gam.predict(XX)

    # 3. Plot
    plt.plot(age_grid, pred_log_price, label=b_name, linewidth=2)

plt.xlabel("Age (Years)")
plt.ylabel("Log Price (Absolute Value)")
plt.title("Sanity Check: Absolute Depreciation Curves")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## **Objective 2 : Mechanical Performance Analysis**

###**EDA**-**Objective2**

####**1.Count of Fuel Types,gear and Engine Segment**

In [ ]:
plt.figure(figsize=(20,6))  # wider figure for one row of plots

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(20,6))

# Gear
ax1 = sns.countplot(x='gear', data=df, palette="pastel", hue='gear', ax=axes[0])
axes[0].set_xlabel("Gear Type")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Gear Types")
for p in ax1.patches:
    ax1.text(p.get_x() + p.get_width()/2., p.get_height() + 0.3, int(p.get_height()), ha='center')

# Fuel Type
ax2 = sns.countplot(x='fuel_type', data=df, palette="pastel", hue='fuel_type', ax=axes[1])
axes[1].set_xlabel("Fuel Type")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Fuel Types")
axes[1].tick_params(axis='x', rotation=45)
for p in ax2.patches:
    ax2.text(p.get_x() + p.get_width()/2., p.get_height() + 0.3, int(p.get_height()), ha='center')

# Engine Segment
ax3 = sns.countplot(x='engine_segment', data=df, palette="pastel", hue='engine_segment', ax=axes[2])
axes[2].set_xlabel("Engine Segment")
axes[2].set_ylabel("Count")
axes[2].set_title("Distribution of Engine Segment")
axes[2].tick_params(axis='x', rotation=45)
for p in ax3.patches:
    ax3.text(p.get_x() + p.get_width()/2., p.get_height() + 0.3, int(p.get_height()), ha='center')

plt.tight_layout()
plt.show()


#####**Cars by Gear Types and Fuel Type**

In [ ]:
group_table = df.groupby(['gear', 'fuel_type']).size().reset_index(name='Count')
display(group_table)

In [ ]:
ax_1 = sns.countplot(x='gear', data=df, palette="pastel", hue="fuel_type")
plt.xlabel("Gear Type")
plt.ylabel("Count")
plt.title("Distribution of Cars by Gear Types and Fuel Type")

for p in ax_1.patches:
    height = p.get_height()
    if height > 0:
        ax_1.text(p.get_x() + p.get_width()/2., height + 0.3, int(height), ha='center')

plt.xticks(rotation=45)

plt.show()

In [ ]:
#using stacked bar chart
ct = pd.crosstab(df['gear'], df['fuel_type'])

ct.plot(kind='bar', stacked=True, figsize=(8,5))
plt.xlabel("Gear Type")
plt.ylabel("Count")
plt.title("Distribution of cars by Gear and Fuel Type")
plt.xticks(rotation=45)
plt.legend(title="Fuel Type")
plt.show()


#####**Cars by Fuel Type and Engine Segment**

In [ ]:
group_table = df.groupby(['fuel_type', 'engine_segment']).size().reset_index(name='Count')
display(group_table)

In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.countplot(x='fuel_type', data=df, palette="pastel", hue="engine_segment")
plt.xlabel("fuel_type")
plt.ylabel("Count")
plt.title('Distribution of Cars by Fuel Type and Engine Segment')

for p in ax.patches:
    height = p.get_height()
    if height >=0:
        ax.text(p.get_x() + p.get_width()/2., height + 0.3, int(height), ha='center')

plt.xticks(rotation=45)

plt.show()

#####**Cars by Fuel Type,Gear Type and Engine Segment**

In [ ]:
ct = pd.crosstab([df['fuel_type'], df['gear']],df['engine_segment'])

plt.figure(figsize=(10,6))

sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')

plt.xlabel("Engine Segment")
plt.ylabel("Fuel Type, Gear")
plt.title("Distribution of Cars by Fuel, Transmission, and Engine Segment")
plt.show()


In [ ]:
g= sns.catplot(x='gear',data=df,hue='engine_segment',col='fuel_type',palette='pastel',kind="count")
g.fig.suptitle("Distribution of Vehicles by Fuel Type,Gear Type and Engine Segment ", y=1.05)

for ax in g.axes.flatten():
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.text(p.get_x() + p.get_width() / 2,height,int(height),ha='center',va='bottom',fontsize=9)

plt.tight_layout()
plt.show()


####**2.Car Distribution by Brand (brand_grouped) and Engine Segment – counts of cars across brands and engine types.**

In [ ]:
ct = pd.crosstab(df['brand_grouped'], df['engine_segment'])

plt.figure(figsize=(10,6))
sns.heatmap(ct,annot=True,fmt='d',cmap='YlGnBu')

plt.xlabel("Engine Segment")
plt.ylabel("Brand")
plt.title("Car Distribution by Brand (brand_grouped)and Engine Segment")
plt.tight_layout()
plt.show()


In [ ]:
#Get top 5 brands(brand_grouped) by total count
top_5_brands = (df['brand_grouped'].value_counts().head(5).index)

# Filter dataframe
df_top5 = df[df['brand_grouped'].isin(top_5_brands)]

In [ ]:
#grouped table for brands
group_table = (df_top5.groupby(['brand_grouped', 'engine_segment']).size().reset_index(name='Count'))

In [ ]:
#Plot
plt.figure(figsize=(12, 6))
ax = sns.countplot(x='brand_grouped',data=df_top5,hue='engine_segment',palette='pastel')
plt.xlabel("Brand (brand_grouped)")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.title("Vehicle Distribution by top 5 Brands (brand_grouped) and Engine Segment")


#Add value labels
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(p.get_x() + p.get_width() / 2,height + 0.3,int(height),ha='center',fontsize=9)

plt.tight_layout()
plt.show()

####**3.Brands(brand_grouped)  by Count and Gear Type**

In [ ]:
#grouped table for brands
group_table = (df.groupby(['brand_grouped', 'gear']).size().reset_index(name='Count'))
display(group_table.style.background_gradient())

In [ ]:
#Plot
plt.figure(figsize=(12, 6))
ax = sns.countplot(x='brand_grouped',data=df,hue='gear',palette='pastel')
plt.xlabel("Gear Type")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.title("Car Distribution by top 5 Brands (brand_grouped) and Gear Type")


#Add value labels
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(p.get_x() + p.get_width() / 2,height + 0.3,int(height),ha='center',fontsize=9)

plt.tight_layout()
plt.show()

######**Top 5 Brands (brand_grouped)  by gear type**

In [ ]:
# Filter dataframe
df_top5 = df[df['brand_grouped'].isin(top_5_brands)]

#grouped table for top 5 brands
group_table = (df_top5.groupby(['brand_grouped', 'gear']).size().reset_index(name='Count'))
display(group_table.style.background_gradient())



In [ ]:
#Plot
plt.figure(figsize=(12, 6))
ax = sns.countplot(x='brand_grouped',data=df_top5,hue='gear',palette='pastel')
plt.xlabel("Brand(brand_grouped)")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.title("Top 5 Car Brands by Count and gear")


#Add value labels
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(p.get_x() + p.get_width() / 2,height + 0.3,int(height),ha='center',fontsize=9)

plt.tight_layout()
plt.show()


####**4.Distribution of Gear Types Across Engine Segments for Top 5 Brands(brand_grouped)**

In [ ]:
# Filter dataframe
df_top5 = df[df['brand_grouped'].isin(top_5_brands)]

#grouped table for top 5 brands
group_table = (df_top5.groupby(['brand_grouped', 'gear','engine_segment']).size().reset_index(name='Count'))
display(group_table.style.background_gradient())

In [ ]:
g= sns.catplot(x='brand_grouped',data=df_top5,hue='gear',col='engine_segment',palette='pastel',kind="count")
g.fig.suptitle("Distribution of Gear Types Across Engine Segments for Top 5 Brands", y=1.05)

for ax in g.axes.flatten():
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.text(p.get_x() + p.get_width() / 2,height,int(height),ha='center',va='bottom',fontsize=9)

plt.tight_layout()
plt.show()

####**5.Distribution of Fuel Types Across the Top 5 Brands (brand_grouped)**

In [ ]:
#grouped table for brands
group_table = (df_top5.groupby(['brand_grouped', 'fuel_type']).size().reset_index(name='Count'))
display(group_table.style.background_gradient())


In [ ]:
#Plot
plt.figure(figsize=(12, 6))
ax = sns.countplot(x='brand_grouped',data=df_top5,hue='fuel_type',palette='pastel')
plt.xlabel("Brand (brand_grouped)")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.title("Distribution of Fuel Types Across the Top 5 Brands (brand_grouped)")


#Add value labels
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(p.get_x() + p.get_width() / 2,height + 0.3,int(height),ha='center',fontsize=9)

plt.tight_layout()
plt.show()


####**6.Distribution of Gear and Fuel Types by Engine Segment for Top 5 Brands**

In [ ]:
#grouped table for top 5 brands
group_table = (df_top5.groupby(['brand_grouped', 'gear','engine_segment']).size().reset_index(name='Count'))

In [ ]:
g= sns.catplot(x='brand_grouped',data=df_top5,hue='gear',col='engine_segment',row='fuel_type',palette='pastel',kind="count")
g.fig.suptitle("Top 5 Brands by Gear Type,Fuel Type and Engine Segment", y=1.05)

for ax in g.axes.flatten():
    for p in ax.patches:
        height = p.get_height()
        if height > 0:
            ax.text(p.get_x() + p.get_width() / 2,height,int(height),ha='center',va='bottom',fontsize=9)

plt.tight_layout()
plt.show()


####**7.Distribution of car prices across fuel types,gear types and engine segments**

#####**1.Distribution of Car prices across different fuel types**




In [ ]:
display(df.groupby('fuel_type')['price'].describe().style.background_gradient())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(data=df,y='fuel_type',x='price',hue='fuel_type',legend=False,palette='Set2',ax=axes[0])
axes[0].set_title("Distribution of car prices across fuel types")

sns.boxplot(data=df,y='fuel_type',x='price',hue='fuel_type',legend=False,palette='Set2',ax=axes[1])
axes[1].set_title("Distribution of car prices across fuel types")

plt.tight_layout()
plt.show()


In [ ]:
# Compute median prices by fuel type
medians = df.groupby('fuel_type')['price'].median().sort_values()

# Order fuel types by median price
fuel_order = medians.index

plt.figure(figsize=(8, 6))
sns.boxplot(data=df,x='price',y='fuel_type',order=fuel_order,palette='Set2',hue='fuel_type')
plt.title("Distribution of Car Prices by Fuel Type")
plt.xlabel("Price (LKR lakhs)")
plt.ylabel("Fuel Type")
plt.show()


In [ ]:
#highest price in hybrid type
df_original[df_original['Price'] == 790]


In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.barplot(x='fuel_type', y='price', data=df, estimator='median', errorbar=None, palette='viridis', hue='fuel_type', legend=False)
plt.title('Median car Price by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Median Price (LKR lakhs)')
plt.xticks(rotation=45)

# Add labels to the bars
for p in ax.patches:
    ax.text(p.get_x() + p.get_width() / 2., p.get_height(), f'{p.get_height():.1f}',
            fontsize=10, color='black', ha='center', va='bottom')

plt.tight_layout()
plt.show()

Although the dataset contains four fuel types (Petrol, Diesel, Hybrid, and Electric), these were consolidated into two broader categories for analysis. Petrol and Diesel vehicles were grouped as Traditional fuel vehicles, while Hybrid and Electric vehicles were grouped as Alternative fuel vehicles. This grouping enables a clearer and more meaningful comparison between conventional and emerging vehicle technologies.

Traditional fuel vehicles = Petrol + Diesel

Alternative fuel vehicles = Hybrid + Electric

In [ ]:
# Create two groups
alt_prices = df[df['fuel_type'].isin(['Hybrid', 'Electric'])]['price']
trad_prices = df[df['fuel_type'].isin(['Petrol', 'Diesel'])]['price']

In [ ]:
plt.figure(figsize=(7,5))
plt.boxplot(
    [alt_prices, trad_prices],
    labels=['Hybrid + Electric(Alternative cars)', 'Petrol + Diesel(Traditional cars)']
)

plt.title('Car Price Comparison: Alternative vs Traditional Fuel Vehicles')
plt.ylabel('Price (LKR lakhs)')
plt.grid(axis='y', alpha=0.3)
plt.show()


#####**2.Distribution of car prices across different gear types**

In [ ]:
display(df.groupby('gear')['price'].describe().style.background_gradient())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(data=df,y='gear',x='price',hue='gear',legend=False,palette='Set2',ax=axes[0])
axes[0].set_title("Distribution of car prices across geartypes")

sns.boxplot(data=df,y='gear',x='price',hue='gear',legend=False,ax=axes[1],palette='Set2')
axes[1].set_title("Distribution of car prices across gear type")

plt.tight_layout()
plt.show()


#####**3.Distribution of car prices across different engine segment types**

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df,
    x='engine_cc',
    y='price',
    alpha=0.6
)

plt.title("Engine Capacity vs Vehicle Price")
plt.xlabel("Engine Capacity (cc)")
plt.ylabel("Price (LKR Lakhs)")

plt.tight_layout()
plt.show()


In [ ]:
display(df.groupby('engine_segment')['price'].describe().style.background_gradient())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.violinplot(data=df,y='engine_segment',x='price',hue='engine_segment',legend=False,palette='Set2',ax=axes[0])
axes[0].set_title("Distribution of car prices across engine_segment types")

sns.boxplot(data=df,y='engine_segment',x='price',hue='engine_segment',legend=False,ax=axes[1],palette='Set2')
axes[1].set_title("Distribution of car prices across engine_segment types")

plt.tight_layout()
plt.show()


In [ ]:
# Order engine segments by median price
engine_order = (
    df.groupby('engine_segment')['price']
      .median()
      .sort_values()
      .index
)

plt.figure(figsize=(8, 6))
sns.boxplot(
    data=df,
    x='price',
    y='engine_segment',
    order=engine_order,
    palette='Set2'
)

plt.title("Distribution of Car Prices by Engine Segment")
plt.xlabel("Price (LKR Lakhs)")
plt.ylabel("Engine Segment")

plt.tight_layout()
plt.show()


#####**4.Distribution of car Price by Fuel Type, Gear, and Engine Segment**

In [ ]:
display(df.groupby(['gear','fuel_type','engine_segment'])['price'].describe().style.background_gradient())

In [ ]:
g = sns.catplot(
    data=df,x='fuel_type',y='price',hue='gear',col='engine_segment',kind='box',height=4,aspect=1)

# Add an overall title
g.fig.suptitle('Car Price Distribution by Fuel Type, Gear, and Engine Segment', fontsize=16)
g.fig.subplots_adjust(top=0.85)  # Adjust to make space for the suptitle

plt.show()


In [ ]:
g=sns.catplot(data=df, x='fuel_type', y='price', hue='gear', col='engine_segment', kind='bar', estimator='median',errorbar=None,
    height=8,aspect=1)
g.fig.suptitle('Median Price by Fuel Type, Gear, and Engine Segment', fontsize=16)
g.fig.subplots_adjust(top=0.85)

plt.show()

###**Advanced Statistical Analysis-objective 2**

####**How Fuel type affects price and whether Alternative fuels (Hybrid and Electric) cars command higher prices than traditional fuels (Petrol and Diesel) cars.**

Research Question:

**Do vehicle prices differ across fuel types (Petrol, Diesel, Hybrid, Electric)?**

try to use, **One-Way ANOVA**

This test is used to see if there is a variation in the mean values of three or more groups. These must be validated before analysis:

**Independence**: Observations are independent.

**Normality**: Residuals are approximately normally distributed (checked via Q-Q plots or Shapiro-Wilk test).

**Homoscedasticity**: Equal variances across groups (verified using Levene's test).



**Check Variance equality (Levene test)**

Levene’s test is used to check whether different groups have equal variance.

**H₀:The variances of prices are equal across fuel types.**

**H₁:At least one fuel type has a different price variance.**


In [ ]:
#significance level
alpha = 0.05

In [ ]:
# Create groups
#You split the data into separate series for each fuel type

petrol = df[df['fuel_type'] == 'Petrol']['price']
diesel = df[df['fuel_type'] == 'Diesel']['price']
hybrid = df[df['fuel_type'] == 'Hybrid']['price']
electric = df[df['fuel_type'] == 'Electric']['price']

In [ ]:
#Levene test
levene_stat, levene_p = levene(petrol, diesel, hybrid, electric,center='median')

print(f'levene_stat={levene_stat:.2f}')
print(f'levene p value ={levene_p:.2f}')

# Decision
alpha = 0.05
if levene_p < alpha:
    print("Decision: Reject the null hypothesis; variances are not equal")
else:
    print("Decision: Fail to reject the null hypothesis;variances are equal")

Therefore one way ANOVA is not appropriate, because it assumes equal variances.

using non parametric test, **Kruskal-Wallis Test**



Now,
Do vehicle prices differ across fuel types (Petrol, Diesel, Hybrid, Electric)?

**Hypotheses (two-sided test):**

- **Null hypothesis (H₀):**  
The median prices of cars are equal across all fuel types (Petrol, Diesel, Hybrid, Electric).  
$$
H_0: \text{Median price}_{\text{Petrol}} = \text{Median price}_{\text{Diesel}} = \text{Median price}_{\text{Hybrid}} = \text{Median price}_{\text{Electric}}
$$

- **Alternative hypothesis (H₁):**  
At least one fuel type has a median price that is different from the others.  
$$
H_1: \text{At least one median price differs among the fuel types}
$$


In [ ]:
kruskal_stat, kruskal_p = kruskal(petrol, diesel, hybrid, electric)

print(f"Kruskal-Wallis statistic = {kruskal_stat:.4f}")
print(f"Kruskal-Wallis p-value   = {kruskal_p:.4f}")

if kruskal_p < 0.05:
    print("Decision: Reject H0 : median prices differ across fuel types")
else:
    print("Decision: Fail to reject H0 : no significant difference")


Kruskal-Wallis only tells us that a difference exists somewhere, but not which fuel types differ.

Dunn's Test(non parametric) is used after the Kruskal-Wallis one-way analysis of variance by ranks to identify which groups differ from each other.



For each pair of fuel types:

Null hypothesis:Median prices of the two fuel types are equal.

Alternative hypothesis:Median prices of the two fuel types are different.

In [ ]:
#this test expects a list of arrays or series, where each element is the data for one group.
groups = [petrol, diesel, hybrid, electric]
group_names = ['Petrol', 'Diesel', 'Hybrid', 'Electric']

In [ ]:
# Run Dunn's test with Bonferroni correction
posthoc = sp.posthoc_dunn(groups, p_adjust='bonferroni')

# Add labels
posthoc.index = group_names
posthoc.columns = group_names

print(posthoc)

In [ ]:
from itertools import combinations

# Prepare summary
pairs = list(combinations(group_names, 2))
m = len(pairs)  # number of comparisons
alpha = 0.05
alpha_adj = alpha / m  # Bonferroni adjusted alpha

summary = []
for g1, g2 in pairs:
    p_val = posthoc.loc[g1, g2]                 # raw p-value
    p_adj = min(p_val * m, 1.0)                  # Bonferroni adjusted p-value
    sig = "Yes" if p_adj < alpha else "No"       # decision using adjusted p-value

    summary.append([
        f"{g1} vs {g2}",
        round(p_val, 4),
        round(p_adj, 4),
        sig
    ])

summary_df = pd.DataFrame(
    summary,
    columns=['Comparison', 'Raw p-value', 'Adjusted p-value', 'Significant?']
)

display(summary_df)

In [ ]:
groups = [petrol, diesel, hybrid, electric]
group_names = ['Petrol', 'Diesel', 'Hybrid', 'Electric']

# Dunn’s test with Holm correction
posthoc = sp.posthoc_dunn(groups, p_adjust='holm')

# Label rows and columns
posthoc.index = group_names
posthoc.columns = group_names

posthoc_rounded = posthoc.round(6)

display(posthoc_rounded)




**Research Question:**  
Do Alternative fuel (Hybrid and Electric) cars command higher (median) prices than Traditional fuel (Petrol and Diesel) cars?

**Hypotheses (one-sided test):**

- **Null hypothesis (H₀):**  
Alternative fuel (Hybrid and Electric) cars have higher (median) prices than Traditional fuel (Petrol and Diesel) cars.
$$
H_0: \text{(median) Prices of Alternative fuels} = \text{(median) Prices of Traditional fuels}
$$


- **Alternative hypothesis (H₁):**  
Alternative fuel (Hybrid and Electric) cars have higher prices than Traditional fuel (Petrol and Diesel) cars.
$$
H_1: \text{(median) Prices of Alternative fuels} > \text{(median) Prices of Traditional fuels}
$$


Traditional fuel vehicles = Petrol + Diesel

Alternative fuel vehicles = Hybrid + Electric

In [ ]:
# Create two groups
alt_prices = df[df['fuel_type'].isin(['Hybrid', 'Electric'])]['price']
trad_prices = df[df['fuel_type'].isin(['Petrol', 'Diesel'])]['price']

In [ ]:
# Mann-Whitney U test, one-sided
u_stat, p_value = mannwhitneyu(alt_prices, trad_prices, alternative='greater')

In [ ]:
print("Mann Whitney U =", u_stat)
print("One-sided p-value =", p_value)
print ("\n ")
# Interpretation

alpha = 0.05
print ("At 5% significance level, \n ")

if p_value < alpha:
  print ("p < 0.05 ")
  print ("Reject H0")
  print("Alternative fuel (Hybrid & Electric) cars have significantly higher prices than Traditional fuel(Petrol and Diesel) cars.")
else:
  print ("p ≥ 0.05, \n ")
  print ("Fail to reject H0, \n ")
  print("No evidence that Alternative fuel (Hybrid & Electric) cars are more expensive than Traditional fuel(Petrol and Diesel) cars.")

####**How transmission(gear) type affects price and whether Automatic cars are priced higher than Manual cars due to driving convenience**

In [ ]:
# Split prices by transmission(gear) type
auto = df[df['gear'] == 'Automatic']['price']
manual = df[df['gear'] == 'Manual']['price']

In this case we try to use Independent two-sample t-test (parametric)

It assumes:

Both groups are normally distributed

Groups have equal or roughly equal variances

In [ ]:
# Test normality for prices by transmission(gear) type
stat_auto, p_auto = shapiro(auto)
stat_manual, p_manual = shapiro(manual)

print(f"Auto: W={stat_auto:.3f}, p={p_auto:.4f}")
print(f"Manual: W={stat_manual:.3f}, p={p_manual:.4f}")

# Interpretation
alpha = 0.05
if p_auto < alpha:
    print("Auto group is NOT normally distributed")
else:
    print("Auto group is approximately normal")

if p_manual < alpha:
    print("Manual group is NOT normally distributed")
else:
    print("Manual group is approximately normal")

Both Automatic and Manual car prices are not normally distributed (Shapiro–Wilk p < 0.001).
Therefore, we use the Mann–Whitney U test instead of a t-test to compare prices between Auto and Manual cars

**Hypotheses (one-sided test):**

- **Null hypothesis (H₀):**  
The median price of Automatic cars is **equal** to the median price of Manual cars.  
$$
H_0: \text{Median price}_{\text{Automatic}} = \text{Median price}_{\text{Manual}}
$$

- **Alternative hypothesis (H₁):**  
The median price of Automatic cars is **greater** than the median price of Manual cars.  
$$
H_1: \text{Median price}_{\text{Automatic}} > \text{Median price}_{\text{Manual}}
$$


In [ ]:
# Mann–Whitney U test (one-sided: Automatic > Manual)
u_stat, p_val = mannwhitneyu(auto, manual, alternative='greater')

print(f"Mann-Whitney U statistic = {u_stat}")
print(f"P-value = {p_val:.4f}")

# Interpretation
alpha = 0.05
print("At 5% significance level:\n")

if p_val < alpha:
    print("p < 0.05,")
    print("Reject H0,")
    print("Automatic vehicles have significantly higher prices than Manual vehicles due to driving convenience.")
else:
    print("p ≥ 0.05,")
    print("Fail to reject H0,")
    print("No evidence that Automatic vehicles are more expensive than Manual vehicles.")

####**How engine size affects price and whether it modifies the impact of fuel type or transmission on price.**

**Research Question:**  
Does engine size influence vehicle price?




Visualize the relationship

In [ ]:
sns.scatterplot(x='engine_cc', y='price', data=df)
plt.title("Engine Size vs Vehicle Price")
plt.xlabel("Engine Size")
plt.ylabel("Price")
plt.show()


In this case we try to use **Pearson Correlation Coefficient**,

Asumptions:
Both variables should be continuous

Relationship should be approximately linear

No extreme outliers (or consider robust methods)

Variables roughly normally distributed


In [ ]:
sns.boxplot(x=df['engine_cc'])
plt.title("Engine Size Outliers")
plt.show()

sns.boxplot(x=df['price'])
plt.title("Price Outliers")
plt.show()


It contains Extreme outliers and both variables are notroughly normal.
therefore we use Spearman correlation,

**Hypotheses:**

- **Null hypothesis (H₀):**  
There is **no correlation** between engine size and vehicle price.

- **Alternative hypothesis (H₁):**  
There is a **positive correlation** between engine size and vehicle price; vehicles with **larger engines tend to have higher prices**.

In [ ]:
stat, p = spearmanr(df['engine_cc'], df['price'])
print('Spearman correlation=%.3f, p=%.3f' % (stat, p))

In [ ]:
# Interpretation
alpha = 0.05
print("At 5% significance level:\n")

if p < alpha:
    direction = "positive" if stat > 0 else "negative"
    print(f"p < 0.05, Reject H₀,")
    print(f"There is a significant {direction} correlation between engine size and vehicle price.")
    print(f"Vehicles with larger engines tend to have {direction} prices.")
else:
    print("p ≥ 0.05, Fail to reject H₀,")
    print("No evidence of a correlation between engine size and vehicle price.")

engine size significantly correlates with price

0.088 is very close to 0, which means the relationship between engine size and price is extremely weak.

Even though p < 0.05, the correlation coefficient is so small (0.088) that engine size alone barely explains price differences.


**Research Question:**  
**Do vehicle prices differ across engine types?**

**Hypotheses (Kruskal–Wallis test):**

- **Null hypothesis (H₀):**  
The median vehicle price is the same across all engine types.  
$$
H_0: \text{Median price}_{\text{Engine Type 1}} = \text{Median price}_{\text{Engine Type 2}} = ... = \text{Median price}_{\text{Engine Type n}}
$$

- **Alternative hypothesis (H₁):**  
At least one engine type has a median price that is different from the others.  
$$
H_1: \text{At least one median price differs among engine types}
$$


In [ ]:
segments = df['engine_segment'].unique()
groups = [df[df['engine_segment'] == seg]['price'] for seg in segments]

stat, p = kruskal(*groups)  # Use * to unpack the list
print('Kruskal-Wallis H=%.3f, p=%.3f' % (stat, p))

In [ ]:
alpha = 0.05
print("At 5% significance level:\n")

if p < alpha:
    print("p < 0.05, Reject H₀,")
    print("There is a significant difference in median vehicle prices across engine segments.")
else:
    print("p ≥ 0.05, Fail to reject H₀,")
    print("No evidence of a difference in median vehicle prices across engine segments.")

Compare each pair of engine segments to see which ones have significantly different median prices.

In [ ]:
posthoc = sp.posthoc_dunn(groups, p_adjust='bonferroni')
posthoc.index = segments
posthoc.columns = segments


In [ ]:

# All pairs of engine segments
pairs = list(combinations(segments, 2))

summary = []
alpha = 0.05

for g1, g2 in pairs:
    p_val = posthoc.loc[g1, g2]
    sig = "Yes" if p_val < alpha else "No"
    summary.append([f"{g1} vs {g2}", p_val, sig])

summary_df = pd.DataFrame(summary, columns=['Comparison', 'p-value', 'Significant?'])
display(summary_df)

## **Objective 3:Comfort Feature Analysis**

### 1.Frequency Distribution of Comfort Features

In [ ]:
# Global Styling
sns.set_theme(style="whitegrid", font_scale=1.15)
plt.rcParams.update({
    'axes.titlesize': 17, 'axes.labelsize': 17.5,
    'xtick.labelsize': 12.5, 'ytick.labelsize': 12.5, 'legend.fontsize': 11.5
})

cat_features = ['air_condition', 'power_steering', 'power_mirror', 'power_window']
avail_order = ['Available', 'Not_Available']
avail_palette = ['#4CAF50', '#FF5733']

# Count Plots
fig, axes = plt.subplots(2, 2, figsize=(14.5, 9.5))
for ax, col in zip(axes.flatten(), cat_features):
    sns.countplot(data=df, x=col, hue=col, palette='viridis', legend=False,
                  ax=ax, width=0.42, saturation=0.88)

    ax.set(title=col.replace("_", " ").title(), xlabel='', ylabel='Count')
    for container in ax.containers:
        ax.bar_label(container, fmt='%d', fontsize=11, padding=5)

fig.suptitle('Frequency Distribution of Comfort Features', fontsize=19, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### 2.Comfort Features Impact on Vehical Price

In [ ]:
# Boxplots
fig, axes = plt.subplots(2, 2, figsize=(14.5, 10.5))
for ax, col in zip(axes.flatten(), cat_features):
    sns.boxplot(data=df, x=col, y='price', hue=col, palette=avail_palette,
                hue_order=avail_order, legend=False, ax=ax, width=0.42,
                saturation=0.92, fliersize=3.5)

    ax.set(title=f'Price vs {col.replace("_", " ").title()}', xlabel='', ylabel='Price (LKR)')

fig.suptitle('Comfort Features Impact on Vehicle Price', fontsize=19, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

### 3.Feature Penetration Rate (%) by Year Category

In [ ]:

# Opt-in to future behavior to silence the warning
pd.set_option('future.no_silent_downcasting', True)

#Process data: Map, Group, and Scale
era_features = (
    df[['year_category'] + cat_features]
    .replace({'Available': 1, 'Not_Available': 0})
    .groupby('year_category')[cat_features]
    .mean() * 100
).reindex(['Old', 'Intermediate', 'Modern'])

#Visualization
era_features.plot(kind='bar', figsize=(12, 6), colormap='viridis', edgecolor='black', width=0.8)

plt.title('Feature Penetration Rate (%) by Year Category', fontsize=15, fontweight='bold', pad=20)
plt.ylabel('Percentage of Vehicles (%)')
plt.xlabel('Year Category')
plt.xticks(rotation=0)
plt.legend(title='Comfort Feature', bbox_to_anchor=(1.05, 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:

#Strip Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True, sharey=True)
era_order = ["Old", "Intermediate", "Modern"]

for ax, feature in zip(axes.flatten(), cat_features):
    sns.stripplot(
        data=df, x="year_category", y="price", hue=feature, ax=ax,
        order=era_order, dodge=True, alpha=0.5, palette="Set1",
        size=3.8, jitter=0.2
    )

    ax.xaxis.set_tick_params(labelbottom=True)
    ax.xaxis.get_label().set_visible(True)

    ax.set_title(feature.replace("_", " ").title(), fontweight='semibold')
    ax.set(xlabel="", ylabel="Price (LKR)")
    ax.legend(title=None, loc="upper right", fontsize=8.5)

fig.suptitle("Individual Vehicle Prices by Year Category and Feature", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.Cramér’s V Heatmap for Comfort Features

In [ ]:
def cramers_v(x, y):
    #Calculates bias-corrected Cramér's V.
    cm = pd.crosstab(x, y)
    chi2 = chi2_contingency(cm)[0]
    n = cm.sum().sum()
    r, k = cm.shape

    phi2 = max(0, chi2/n - ((k-1)*(r-1))/(n-1))
    rcorr = r - ((r-1)**2)/(n-1)
    kcorr = k - ((k-1)**2)/(n-1)
    return np.sqrt(phi2 / min((kcorr-1), (rcorr-1)))

# Prepare Matrix
features = ['air_condition', 'power_steering', 'power_mirror', 'power_window']
matrix_data = [[cramers_v(df[c1], df[c2]) for c2 in features] for c1 in features]
cramers_matrix = pd.DataFrame(matrix_data, index=features, columns=features)

# Plotting Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(cramers_matrix, annot=True, fmt=".2f", cmap='YlGnBu', vmin=0, vmax=1, square=True)

plt.title("Cramér’s V Heatmap for Comfort Features", fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

### 5.Average Price vs Year of Manufacture by Comfort Features

In [ ]:
#Line Plots
sns.set_theme(style="whitegrid")

features = ['air_condition', 'power_steering', 'power_mirror', 'power_window']
fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True, sharey=True)

for ax, feature in zip(axes.flatten(), features):
    nice_name = feature.replace('_', ' ').title()

    sns.lineplot(
        data=df, x='yom', y='price', hue=feature,
        marker='o', errorbar=None, palette='magma',
        linewidth=2.2, markersize=7, ax=ax
    )

    # Force x-axis labels and ticks to stay visible
    ax.xaxis.set_tick_params(labelbottom=True)
    ax.xaxis.get_label().set_visible(True)

    # Formatting
    ax.set_title(f'Price vs Year by {nice_name}', fontweight='bold', pad=10)
    ax.set(xlabel='Year of Manufacture (YOM)', ylabel='Average Price (LKR)')
    ax.legend(title=nice_name, loc='upper left', framealpha=0.9)
    ax.grid(True, linestyle='--', alpha=0.5)

fig.suptitle('Average Price vs Year of Manufacture by Comfort Features', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.Testing Normality and the Homogenity of Variances within Availability of Comfort Features
Before comparing prices between vehicles with and without comfort features, we assessed:

1. **Normality of Price**  
   - **Test used:** D’Agostino K² (as the Shapiro-Wilk test is unreliable for large datasets)  
   - **Hypotheses:**  
     - H₀: Price is normally distributed  
     - H₁: Price is not normally distributed  

2. **Homogeneity of Variance (Levene’s Test) per Comfort Feature**  
   - **Test used:** Levene’s test (median-centered, robust to non-normality)  
   - **Features tested:** Air Conditioning, Power Steering, Power Mirror, Power Window  
   - **Hypotheses:**  
     - H₀: Variance of price is equal between vehicles **Available** and **Not Availble** the feature  
     - H₁: Variance of price is unequal between vehicles **Available** and **Not Availble** the feature  



In [ ]:
FEATURES = ['air_condition', 'power_steering', 'power_mirror', 'power_window']
ALPHA = 0.05

results = []

# Normality for Price(D'Agostino K²)
price_clean = df['price'].dropna()
stat_norm, p_norm = normaltest(price_clean)

results.append({
    'Test': "Normality (D'Agostino K²)",
    'Variable': "Price",
    'Statistic': stat_norm,
    'p-value': p_norm,
    'Conclusion': "Not Normal" if p_norm < ALPHA else "Normal"
})


In [ ]:
# Levene’s Test per Comfort Feature

for feature in FEATURES:

    nice_name = feature.replace('_', ' ').title()

    # Consistent sample: feature + price must exist
    valid = df[[feature, 'price']].dropna()
    levels = valid[feature].unique()

    # Must be binary
    if len(levels) != 2:
        results.append({
            'Test': "Levene’s Test",
            'Variable': nice_name,
            'Statistic': None,
            'p-value': None,
            'Conclusion': f"Skipped (not binary: {len(levels)} levels)"
        })
        continue

    levels = sorted(levels)
    group_0 = valid[valid[feature] == levels[0]]['price']
    group_1 = valid[valid[feature] == levels[1]]['price']

    if min(len(group_0), len(group_1)) < 10:
        results.append({
            'Test': "Levene’s Test",
            'Variable': nice_name,
            'Statistic': None,
            'p-value': None,
            'Conclusion': "Skipped (too few observations)"
        })
        continue

    stat_lev, p_lev = levene(group_0, group_1, center='median')

    results.append({
        'Test': "Levene’s Test",
        'Variable': nice_name,
        'Statistic': stat_lev,
        'p-value': p_lev,
        'Conclusion': "Unequal Variances" if p_lev < ALPHA else "Equal Variances"
    })

In [ ]:

# Results Table

results_df = pd.DataFrame(results)

results_df['Statistic'] = results_df['Statistic'].map(
    lambda x: f"{x:,.4f}" if pd.notnull(x) else ""
)
results_df['p-value'] = results_df['p-value'].map(
    lambda x: f"{x:.4e}" if pd.notnull(x) else ""
)

styled_results = (
    results_df.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('background-color', '#f2f2f2'),
                ('font-weight', 'bold'),
                ('text-align', 'left'),
                ('padding', '8px')
            ]
        },
        {
            'selector': 'td',
            'props': [('padding', '6px')]
        }
    ])
)

styled_results

### 7.Mann–Whitney U test (two-sided)

To test whether vehicle prices differ between **Available vs Not Available** for each comfort feature, we used the **Mann–Whitney U test** (non-parametric, two-sided) because price is not normally distributed.

**Features tested:** Air Conditioning, Power Steering, Power Mirror, Power Window  
**Significance level:** α = 0.05

**Hypotheses:**

- **H₀:** There is no significant difference in the **median price** between vehicles with feature *i* available and those without feature *i* available.  
- **H₁:** There is a statistically significant difference in the **median price** between vehicles with feature *i* available and those without feature *i* available.



In [ ]:

FEATURES = ['air_condition', 'power_steering', 'power_mirror', 'power_window']
ALPHA = 0.05

results = []

for feature in FEATURES:

    nice_name = feature.replace('_', ' ').title()

    # Consistent sample: feature + price must both exist
    valid = df[[feature, 'price']].dropna()
    levels = valid[feature].unique()

    # Must be binary
    if len(levels) != 2:
        results.append({
            'Test': 'Mann–Whitney U',
            'Variable': nice_name,
            'Statistic': None,
            'p-value': None,
            'Conclusion': f'Skipped (not binary: {len(levels)} levels)'
        })
        continue

    # Split groups
    levels = sorted(levels)
    g0 = valid[valid[feature] == levels[0]]['price']
    g1 = valid[valid[feature] == levels[1]]['price']

    # Minimum sample size check
    if min(len(g0), len(g1)) < 10:
        results.append({
            'Test': 'Mann–Whitney U',
            'Variable': nice_name,
            'Statistic': None,
            'p-value': None,
            'Conclusion': 'Skipped (too few observations)'
        })
        continue

    # Mann–Whitney U test (two-sided)
    stat, p_val = mannwhitneyu(g0, g1, alternative='two-sided')

    results.append({
        'Test': 'Mann–Whitney U',
        'Variable': nice_name,
        'Statistic': stat,
        'p-value': p_val,
        'Conclusion': 'Significant' if p_val < ALPHA else 'Not Significant'
    })


In [ ]:
# Results table
results_df = pd.DataFrame(results)

results_df[['Statistic', 'p-value']] = results_df[['Statistic', 'p-value']].apply(
    lambda s: s.map(lambda x: f'{x:,.0f}' if s.name == 'Statistic' and pd.notnull(x)
                    else f'{x:.4e}' if s.name == 'p-value' and pd.notnull(x)
                    else '')
)

styled_df = (
    results_df.style
    .hide(axis='index')
    .set_properties(**{'text-align': 'left'})
    .set_table_styles([
        {'selector': 'th',
         'props': [('background-color', '#f0f0f0'),
                   ('font-weight', 'bold'),
                   ('text-align', 'left'),
                   ('padding', '8px')]},
        {'selector': 'td', 'props': [('padding', '6px')]}
    ])
)

styled_df

## **Objective 4:Market Sentiment and Geo-Economic Analysis**

###**EDA**-**Objective 4**

####**1.Distribution of Vehicles (Listings) Across Urban and Non-Urban Areas**

In [ ]:
# Count listings by location type
loc_counts = df["location_type"].value_counts()

# Bar plot
plt.figure(figsize=(6,4))
bars= plt.bar(loc_counts.index, loc_counts.values,color=['orange','green'])
plt.title("Distribution of Vehicles (Listings) Across Urban and Non-Urban Areas",y=1.01)
plt.xlabel("Location Type")
plt.ylabel("Number of Vehicles (Listings)")
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{int(height)}",
        ha="center",
        va="bottom"
    )
plt.tight_layout()
plt.show()


####**2.Number of Vehicles (listings) by leasing availability**

In [ ]:
plt.figure(figsize=(6,4))
leasing_count= df['leasing'].value_counts()

bars=plt.bar(leasing_count.index,leasing_count.values,color=['maroon','blue'])
plt.title('Number of Vehicles (listings) by leasing availability')
plt.xlabel('Leasing Availability')
plt.ylabel('Number of Listings')
for bar in bars:
    height=bar.get_height()
    plt.text(
        bar.get_x()+bar.get_width()/2,
        height,
        f"{int(height)}",
        ha='center',
        va='bottom'
    )

In [ ]:
top_towns = df["town"].value_counts().head(15)

plt.figure(figsize=(10,6))
bars= plt.barh(top_towns.index, top_towns.values)
plt.gca().invert_yaxis()
plt.title("Top 15 Towns by Number of Listings")
plt.xlabel("Number of Listings")
plt.ylabel("Town")

for bar in bars:
    width = bar.get_width()
    plt.text(
        width,
        bar.get_y() + bar.get_height()/2,
        f"{int(width)}",
        ha="left",
        va="center"
    )
plt.tight_layout()
plt.show()


In [ ]:
prov_counts = df["province"].value_counts()

plt.figure(figsize=(8,5))
bars = sns.barplot(x=prov_counts.index, y=prov_counts.values, palette='Set2')

for b in bars.patches:
    plt.text(b.get_x()+b.get_width()/2, b.get_height(), int(b.get_height()),
             ha="center", va="bottom")

plt.title("Number of Listings by Province")
plt.xlabel("Province")
plt.xticks(rotation=45, ha='right')

plt.ylabel("Listings")
plt.tight_layout()
plt.show()


In [ ]:
# Calculate average price by YOM and location type
price_by_yom = df.groupby(['yom', 'location_type'])['price'].mean().unstack()

# Plot
plt.figure(figsize=(12, 6))
# Plot each series separately, dropping NaNs to ensure lines connect
for column in price_by_yom.columns:
    data = price_by_yom[column].dropna()
    plt.plot(data.index, data.values, marker='o', markersize=4, label=column)

plt.title('Average Vehicle Price vs Year of Manufacture by Location Type')
plt.xlabel('Year of Manufacture (YOM)')
plt.ylabel('Average Price')
plt.legend(title='Location Type')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate average price by YOM and leasing availability
price_by_yom_leasing = df.groupby(['yom', 'leasing'])['price'].mean().unstack()

# Plot
plt.figure(figsize=(12, 6))
# Plot each series separately, dropping NaNs to ensure lines connect
for column in price_by_yom_leasing.columns:
    data = price_by_yom_leasing[column].dropna()
    plt.plot(data.index, data.values, marker='o', markersize=4, label=column)

plt.title('Average Vehicle Price vs Year of Manufacture by Leasing Availability')
plt.xlabel('Year of Manufacture (YOM)')
plt.ylabel('Average Price')
plt.legend(title='Leasing')
plt.grid(True)
plt.tight_layout()
plt.show()

####**3.Price and Vehicles Listings Summary by Location Type and Leasing Status**

In [ ]:
summary = df.groupby(["location_type", "leasing"]).agg(
    Listings=("price", "count"),
    MeanPrice=("price", "mean"),
    MedianPrice=("price", "median")
)

summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

#Overall price distribution
axes[0].hist(df["price"].dropna(), bins=50)
axes[0].set_title("Overall Price Distribution")
axes[0].set_xlabel("Price")
axes[0].set_ylabel("Frequency")

#Price by location
df.boxplot(column="price", by="location_type", ax=axes[1])
axes[1].set_title("Price by Location")
axes[1].set_xlabel("Location Type")
axes[1].set_ylabel("Price")

# Price by leasing
df.boxplot(column="price", by="leasing", ax=axes[2])
axes[2].set_title("Price by Leasing")
axes[2].set_xlabel("Leasing Availability")
axes[2].set_ylabel("Price")

# Remove the automatic pandas suptitle
plt.suptitle("")
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(8,6))
sns.boxplot(
    data=df,
    x="location_type",
    y="price",
    hue="leasing",
    palette=["orange","green"]
)

plt.title("Price Distribution by Location Type and Leasing Status")
plt.xlabel("Location Type")
plt.ylabel("Price")
plt.yscale('log')  # optional: log scale if prices are skewed
plt.tight_layout()
plt.show()


####**4. Advance analysis of objective 4**

**Checking the influence of two variables on price based on complete dataset**

## Bootstrap Analysis (Location)

Perform 10,000 bootstrap iterations to estimate the distribution of the difference in median prices between Urban and Non-Urban listings and calculate the 95% Confidence Interval.

**Reasoning**: Perform bootstrap analysis to estimate the difference in median prices between Urban and Non-Urban listings and calculate the 95% confidence interval

In [ ]:
import numpy as np
from sklearn.utils import resample

# Extract prices
urban_prices = df[df['location_type'] == 'Urban']['price'].dropna().values
non_urban_prices = df[df['location_type'] == 'Non-Urban']['price'].dropna().values

# Bootstrap settings
n_iterations = 10000
bootstrap_diffs = []

# Perform bootstrapping
np.random.seed(42) # For reproducibility
for _ in range(n_iterations):
    # Resample with replacement
    urban_sample = np.random.choice(urban_prices, size=len(urban_prices), replace=True)
    non_urban_sample = np.random.choice(non_urban_prices, size=len(non_urban_prices), replace=True)

    # Calculate difference in medians
    diff = np.median(urban_sample) - np.median(non_urban_sample)
    bootstrap_diffs.append(diff)

# Calculate 95% Confidence Interval
lower_ci = np.percentile(bootstrap_diffs, 2.5)
upper_ci = np.percentile(bootstrap_diffs, 97.5)
mean_diff = np.mean(bootstrap_diffs)

print(f"95% Confidence Interval for Difference in Medians (Urban - Non-Urban): ({lower_ci:.4f}, {upper_ci:.4f})")
print(f"Mean Bootstrapped Difference: {mean_diff:.4f}")

## Bootstrap Analysis (Leasing)
Perform bootstrap analysis to estimate the difference in median prices between Leasing and No-Leasing listings.

**Reasoning**:
Perform bootstrap analysis for Leasing vs. No-Leasing prices to estimate the difference in medians and calculate the 95% confidence interval.


In [ ]:
# Extract prices for leasing and no leasing
lease_prices = df[df['leasing'] == True]['price'].dropna().values
no_lease_prices = df[df['leasing'] == False]['price'].dropna().values

# Bootstrap settings
n_iterations = 10000
bootstrap_diffs_lease = []

# Perform bootstrapping
np.random.seed(42) # For reproducibility
for _ in range(n_iterations):
    # Resample with replacement
    lease_sample = np.random.choice(lease_prices, size=len(lease_prices), replace=True)
    no_lease_sample = np.random.choice(no_lease_prices, size=len(no_lease_prices), replace=True)

    # Calculate difference in medians
    diff = np.median(lease_sample) - np.median(no_lease_sample)
    bootstrap_diffs_lease.append(diff)

# Calculate 95% Confidence Interval
lower_ci_lease = np.percentile(bootstrap_diffs_lease, 2.5)
upper_ci_lease = np.percentile(bootstrap_diffs_lease, 97.5)
mean_diff_lease = np.mean(bootstrap_diffs_lease)

print(f"95% Confidence Interval for Difference in Medians (Leasing - No Leasing): ({lower_ci_lease:.4f}, {upper_ci_lease:.4f})")
print(f"Mean Bootstrapped Difference: {mean_diff_lease:.4f}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot for Location (Urban vs Non-Urban)
axes[0].hist(bootstrap_diffs, bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[0].axvline(lower_ci, color='red', linestyle='--', label=f'95% CI Lower: {lower_ci:.2f}')
axes[0].axvline(upper_ci, color='red', linestyle='--', label=f'95% CI Upper: {upper_ci:.2f}')
axes[0].axvline(0, color='black', linewidth=2, label='Zero Difference')
axes[0].set_title('Bootstrap Difference in Medians\n(Urban - Non-Urban)')
axes[0].set_xlabel('Difference in Price (LKR Lakhs)')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Plot for Leasing (Leasing vs No Leasing)
axes[1].hist(bootstrap_diffs_lease, bins=50, color='lightgreen', edgecolor='black', alpha=0.7)
axes[1].axvline(lower_ci_lease, color='red', linestyle='--', label=f'95% CI Lower: {lower_ci_lease:.2f}')
axes[1].axvline(upper_ci_lease, color='red', linestyle='--', label=f'95% CI Upper: {upper_ci_lease:.2f}')
axes[1].axvline(0, color='black', linewidth=2, label='Zero Difference')
axes[1].set_title('Bootstrap Difference in Medians\n(Leasing - No Leasing)')
axes[1].set_xlabel('Difference in Price (LKR Lakhs)')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

#Inference Modeling

In [ ]:
df.info()

Linear Regression Analysis (OLS) for Car Price

In [ ]:
import pandas as pd
import statsmodels.api as sm

# 1. Select the relevant features
# We exclude 'town' because it likely has too many unique values (high cardinality),
# which would make the model messy. 'province' is a better location feature.
feature_cols = [
    'engine_cc', 'gear', 'fuel_type', 'leasing',
    'air_condition', 'power_steering', 'power_mirror', 'power_window',
    'age', 'brand_grouped', 'province'
]

# 2. Define X (features) and y (target)
X = df[feature_cols].copy()
y = df['price']

# 3. Convert Categorical variables to Numeric (One-Hot Encoding)
# drop_first=True is crucial for OLS to avoid "Perfect Multicollinearity"
# (e.g., if you have 'Manual' and 'Auto', you only need one column 'Is_Auto')
X = pd.get_dummies(X, drop_first=True)

# 4. Handle Boolean columns if they aren't automatically converted
# (Ensures True/False becomes 1/0)
X = X.astype(float)

# 5. Add the Intercept (Constant)
# Unlike sklearn, statsmodels requires you to explicitly add the constant column (b0)
X = sm.add_constant(X)

# 6. Fit the Model
model = sm.OLS(y, X).fit()

# 7. Print the Results
print(model.summary())

residual analysis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import numpy as np

# Prepare values
fitted_vals = model.fittedvalues
residuals = model.resid
influence = model.get_influence()
(c, p) = influence.cooks_distance

# Create 2x2 canvas
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Residuals vs Fitted
sns.scatterplot(x=fitted_vals, y=residuals, ax=axes[0, 0], alpha=0.5)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')

# 2. QQ Plot
sm.qqplot(residuals, line='45', fit=True, ax=axes[0, 1])
axes[0, 1].set_title('QQ Plot of Residuals')

# 3. Histogram of Residuals
sns.histplot(residuals, kde=True, bins=30, ax=axes[1, 0])
axes[1, 0].set_title('Histogram of Residuals')
axes[1, 0].set_xlabel('Residuals')

# 4. Cook’s Distance
axes[1, 1].stem(np.arange(len(c)), c, markerfmt=",")
axes[1, 1].axhline(4/len(c), color='red', linestyle='--', label='4/n threshold')
axes[1, 1].set_title("Cook's Distance (Influential Points)")
axes[1, 1].set_xlabel('Observation Index')
axes[1, 1].set_ylabel("Cook's Distance")
axes[1, 1].legend()

plt.tight_layout()
plt.show()


In [ ]:
import scipy.stats as stats
import pandas as pd
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor

# --- Normality: Shapiro-Wilk ---
shapiro_stat, shapiro_p = stats.shapiro(residuals)

# --- Homoscedasticity: Breusch-Pagan ---
bp_test = het_breuschpagan(residuals, X)
bp_stat, bp_p = bp_test[0], bp_test[1]

# --- Multicollinearity: VIF ---
X_for_vif = X.drop(columns='Intercept', errors='ignore')
vif_df = pd.DataFrame({
    "Feature": X_for_vif.columns,
    "VIF": [variance_inflation_factor(X_for_vif.values, i)
            for i in range(X_for_vif.shape[1])]
}).sort_values("VIF", ascending=False)

# --- Summary table ---
diagnostics_df = pd.DataFrame({
    "Test": [
        "Shapiro-Wilk (Normality)",
        "Breusch-Pagan (Homoscedasticity)"
    ],
    "Statistic": [
        round(shapiro_stat, 4),
        round(bp_stat, 4)
    ],
    "p-value": [
        round(shapiro_p, 5),
        round(bp_p, 5)
    ],
    "Conclusion": [
        "Normal residuals" if shapiro_p > 0.05 else "Reject normality",
        "Homoscedastic" if bp_p > 0.05 else "Heteroscedastic"
    ]
})

print("\nSTATISTICAL DIAGNOSTIC TESTS")
print("="*60)
print(diagnostics_df.to_string(index=False))
print("="*60)

print("\nVARIANCE INFLATION FACTORS")
print("="*60)
print(vif_df.to_string(index=False))
print("="*60)


Linear Regression Analysis (OLS) for Log-Transformed Car Price

In [ ]:
import statsmodels.api as sm
from patsy import dmatrices

# 1. Define Target
target = 'price_transformed'

# 2. Define columns you explicitly want to ignore
# (I added the duplicate 'fuel_code' and 'location_type_code' you saw to this list just in case)
exclude = {
    'price', 'price_transformed', 'millage_km', 'location_type', 'yom',
    'simple_model', 'brand_model', 'model_average', 'engine_segment',
    'engine_cc', 'model', 'town', 'brand', 'year_category', 'condition',
    'millage_scaled','mileage_resid'
}

# 3. Select Predictors
# Logic: Keep column IF it is not in exclude list AND it does not end with "_code"
predictors = [
    col for col in df.columns
    if col not in exclude and not col.endswith('_code')
]

# 4. Build Formula
formula = f"{target} ~ {' + '.join(predictors)}"
print(f"Model Formula: {formula}") # Print this to verify only clean columns are used!

# 5. Fit OLS
y, X = dmatrices(formula, data=df, return_type='dataframe')
model_log = sm.OLS(y, X).fit()

print(model_log.summary())

residual analysis

In [ ]:
# --- 2. PLOTTING (2x2 Grid) ---
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

TITLE_SIZE = 18
LABEL_SIZE = 14
TICK_SIZE = 12

# Plot 1: Residuals vs Fitted
sns.scatterplot(x=fitted_vals, y=residuals, alpha=0.5, ax=axes[0, 0])
axes[0, 0].axhline(0, color='red', linestyle='--', linewidth=2)
axes[0, 0].set_xlabel('Fitted Values', fontsize=LABEL_SIZE)
axes[0, 0].set_ylabel('Residuals', fontsize=LABEL_SIZE)
axes[0, 0].set_title('1. Residuals vs Fitted (Linearity check)', fontsize=TITLE_SIZE)
axes[0, 0].tick_params(axis='both', labelsize=TICK_SIZE)

# Plot 2: QQ Plot
sm.qqplot(residuals, line='45', fit=True, ax=axes[0, 1])
axes[0, 1].set_title('2. QQ Plot (Normality check)', fontsize=TITLE_SIZE)
axes[0, 1].tick_params(axis='both', labelsize=TICK_SIZE)

# Plot 3: Histogram of Residuals
sns.histplot(residuals, kde=True, bins=30, ax=axes[1, 0])
axes[1, 0].set_title('3. Histogram of Residuals', fontsize=TITLE_SIZE)
axes[1, 0].set_xlabel('Residuals', fontsize=LABEL_SIZE)
axes[1, 0].set_ylabel('Frequency', fontsize=LABEL_SIZE)
axes[1, 0].tick_params(axis='both', labelsize=TICK_SIZE)

# Plot 4: Cook's Distance
axes[1, 1].stem(np.arange(len(c)), c, markerfmt=",", linefmt='C0-')
axes[1, 1].set_xlabel('Observation Index', fontsize=LABEL_SIZE)
axes[1, 1].set_ylabel("Cook's Distance", fontsize=LABEL_SIZE)
axes[1, 1].set_title("4. Influential Points (Cook's Distance)", fontsize=TITLE_SIZE)
axes[1, 1].tick_params(axis='both', labelsize=TICK_SIZE)

plt.tight_layout()
plt.show()


#GAM

In [ ]:
df.info()

In [ ]:
pip install pygam

In [ ]:
import pandas as pd
import numpy as np
from pygam import LinearGAM, s, f, l

# 1. Define Features
continuous_cols = ['log_engine_cc', 'age']
# distinct lists for categorical vs binary helps optimize the model
categorical_cols = ['gear', 'fuel_type', 'province', 'brand_grouped']
binary_cols = ['leasing', 'air_condition', 'power_steering', 'power_mirror', 'power_window']

# 2. Safe Preprocessing (Save the mappings!)
# We keep the original dataframe clean and create a specific 'X_gam' for the model
X_gam = pd.DataFrame()

# Add continuous variables
# Ensure log_engine_cc exists
if 'log_engine_cc' not in df.columns:
    df['log_engine_cc'] = np.log1p(df['engine_cc'])

X_gam[continuous_cols] = df[continuous_cols]

# Add Categorical (and save mappings for later!)
cat_mappings = {}
for col in categorical_cols:
    # Convert to category type if not already
    cat_col = df[col].astype('category')
    # Save the mapping: Code -> String
    cat_mappings[col] = dict(enumerate(cat_col.cat.categories))
    # Assign codes to X_gam
    X_gam[col] = cat_col.cat.codes

# Add Binary (No need for f(), treat as linear/numeric 0s and 1s)
for col in binary_cols:
    # Ensure they are 0 and 1
    X_gam[col] = df[col].astype('category').cat.codes

# 3. Construct the GAM Terms
# s() for continuous (smooth)
# f() for categorical (factor)
# l() for binary (linear) - simpler and faster than f() for Yes/No data

terms = s(0) + s(1) # log_engine_cc and age

# Add categorical terms (starting index is 2)
current_idx = 2
for _ in categorical_cols:
    terms += f(current_idx)
    current_idx += 1

# Add binary terms (linear)
for _ in binary_cols:
    terms += l(current_idx)
    current_idx += 1

# 4. Fit the Model
X = X_gam.values
y = df['price_transformed'].values

gam = LinearGAM(terms).fit(X, y)

# 5. Summary
gam.summary()

# Example: How to check what 'Level 3' of 'gear' means later:
print("Mapping for Gear:", cat_mappings['gear'])

### Generalized Additive Model (GAM): Specification, Diagnostics, and Interpretation

This document describes the estimation, diagnostics, and interpretation of a Generalized Additive Model (GAM) used to analyze vehicle prices. The response variable is log-transformed to stabilize variance and allow multiplicative interpretation.

---

## Model specification

We fit a GAM where log-transformed price is modeled as a smooth, potentially non-linear function of engine capacity and vehicle age, while controlling for brand effects. Smoothing parameters are estimated using REML for stable inference.

```r
gam_model_v2 <- gam(
  price_transformed ~ s(log_engine_cc, k = 20) + s(age, k = 20) +
    gear + fuel_type + leasing +
    air_condition + power_steering + power_mirror +
    power_window + brand_grouped + province,
  data = df,
  method = "REML"
)
```

---

## Model diagnostics

### Basis adequacy and residual diagnostics

The following diagnostic checks ensure that the basis dimensions are sufficiently large and that residual patterns do not indicate model misspecification.

```r
gam.check(gam_model_v2)
```

---

## Visualization of smooth terms

The smooth effects of log(engine capacity) and age are visualized below. Partial residuals are overlaid to assess the adequacy of the fitted smooths.

```r
plot(
  gam_model_v2,
  pages = 1,
  residuals = TRUE,
  pch = 1,
  cex = 0.5,
  scheme = 1
)
```

The diagnostic check is repeated after visual inspection to confirm that the wiggliness limits were appropriate.

```r
gam.check(gam_model_v2)
```

---

## Interpreting smooth effects on the original price scale

Because the response is log-transformed, the inverse transformation is defined to express effects on the original price scale.

```r
unlog_price <- function(x) {
  exp(x)
}
```

The smooth terms are plotted on the real price scale by shifting the curves by the intercept and applying the inverse log transformation.

```r
plot(
  gam_model_v2,
  pages = 1,
  scheme = 1,
  shade = TRUE,
  shade.col = "lightblue",
  shift = coef(gam_model_v2)[1],
  trans = unlog_price,
  seWithMean = TRUE,
  main = "Effect of Engine Capacity and Age on Price"
)
```

---

## Concurvity assessment

Concurvity is examined to assess non-linear dependence between predictors, which is the GAM analogue of multicollinearity.

Overall concurvity:

```r
concurvity(gam_model_v2, full = TRUE)
```

Pairwise concurvity estimates:

```r
cc <- concurvity(gam_model_v2, full = FALSE)
cc$estimate
```

---

## Model summary

```r
summary(gam_model_v2)
```

The summary reports parametric coefficients, smooth term significance, estimated degrees of freedom, and overall model fit statistics.

---

## Interpretation of parametric effects

For parametric (linear) terms, log-scale coefficients are converted into percentage changes for interpretability.

```r
coeffs <- summary(gam_model_v2)$p.table
estimates <- coeffs[, "Estimate"]
percent_impact <- (exp(estimates) - 1) * 100

results_table <- data.frame(
  Variable = names(estimates),
  Log_Coefficient = round(estimates, 3),
  Percent_Change = round(percent_impact, 1)
)

print(results_table)
```

The percentage change values represent the expected percentage change in price associated with each parametric effect.


In [ ]:
# Create a single list of all features in the EXACT order they went into X
all_features = continuous_cols + categorical_cols + binary_cols

# Print the mapping
print(f"{'Index':<10} {'Function':<10} {'Feature Name'}")
print("-" * 40)

for i, name in enumerate(all_features):
    # Determine function type based on which list the feature is in
    if name in continuous_cols:
        func = f"s({i})"
    elif name in categorical_cols:
        func = f"f({i})"
    else:
        func = f"l({i})"

    print(f"{i:<10} {func:<10} {name}")

In [ ]:
fitted_vals = gam.predict(X)
residuals = y - fitted_vals

plt.figure(figsize=(8,5))
sns.scatterplot(x=fitted_vals, y=residuals, alpha=0.5)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel("Fitted values")
plt.ylabel("Residuals")
plt.title("Residuals vs Fitted")
plt.show()


In [ ]:
import statsmodels.api as sm

sm.qqplot(residuals, line='45', fit=True)
plt.title("QQ Plot of Residuals")
plt.show()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(residuals, bins=30, kde=True)
plt.title("Histogram of Residuals")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pygam import LinearGAM, s, f

# Ensure 'log_engine_cc' is present in the DataFrame
if 'log_engine_cc' not in df.columns:
    df['log_engine_cc'] = np.log1p(df['engine_cc'])

# List of all features that will be used in the GAM, in order.
# Continuous features first, then categorical/binary.
continuous_features = ['log_engine_cc', 'age']
categorical_features = [
    'gear', 'fuel_type', 'province', 'leasing',
    'air_condition', 'power_steering', 'power_mirror',
    'power_window', 'brand_grouped'
]

# Create a temporary DataFrame to hold the numerically encoded features
X_temp_df = df[continuous_features].copy()

# Store mappings for categorical columns for later use in plotting labels
cat_mappings = {}

# Encode categorical and binary features to numerical codes
for col in categorical_features:
    cat_series = df[col].astype('category')
    cat_mappings[col] = dict(enumerate(cat_series.cat.categories))
    X_temp_df[col] = cat_series.cat.codes

# Prepare X (features) and y (target)
X = X_temp_df.values
y = df['price_transformed'].values  # log-transformed target

# Build GAM: smooth (s) for continuous, factor (f) for categorical/binary
# The indices correspond to the column positions in the X array.
# s(0) for log_engine_cc, s(1) for age
terms = s(0) + s(1)

# Add factor terms for the rest of the features (starting from index 2)
for i in range(len(continuous_features), X.shape[1]):
    terms += f(i)

# Fit the GAM model
gam = LinearGAM(terms).fit(X, y)

# --- Partial dependence plots for continuous variables ---

# Engine size effect
XX_engine = gam.generate_X_grid(term=0)  # grid for log_engine_cc (index 0)
pdep_engine, conf_engine = gam.partial_dependence(term=0, X=XX_engine, width=0.95)

plt.figure(figsize=(8,5))
plt.plot(XX_engine[:,0], pdep_engine, label='Engine effect')
plt.fill_between(XX_engine[:,0], conf_engine[:,0], conf_engine[:,1], alpha=0.25)
plt.axhline(0, linestyle='--', color='black')
plt.xlabel('Log Engine CC')
plt.ylabel('Effect on log(price)')
plt.title('Smooth Effect of Engine Size')
plt.show()


# Age effect
XX_age = gam.generate_X_grid(term=1)  # grid for age (index 1)
pdep_age, conf_age = gam.partial_dependence(term=1, X=XX_age, width=0.95)

plt.figure(figsize=(8,5))
plt.plot(XX_age[:,1], pdep_age, label='Age effect', color='orange')
plt.fill_between(XX_age[:,1], conf_age[:,0], conf_age[:,1], alpha=0.25, color='orange')
plt.axhline(0, linestyle='--', color='black')
plt.xlabel('Age (years)')
plt.ylabel('Effect on log(price)')
plt.title('Smooth Effect of Age')
plt.show()

Partial Dependence Plots (PDPs)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pygam import LinearGAM, s, f, l

# ------------------------------
# 1. Preprocessing & Outlier Removal
# ------------------------------
df = df[df['price'] < 800].copy()  # remove extreme outlier

if 'log_engine_cc' not in df.columns:
    df['log_engine_cc'] = np.log1p(df['engine_cc'])

# ------------------------------
# 2. Feature Groups
# ------------------------------
continuous_cols = ['log_engine_cc', 'age']
categorical_cols = ['gear', 'fuel_type', 'province', 'brand_grouped']
binary_cols = ['leasing', 'air_condition', 'power_steering', 'power_mirror', 'power_window']

# ------------------------------
# 3. Custom Reference Levels
# ------------------------------
cat_ref = {
    'brand_grouped': 'TOYOTA',
    'gear': 'Manual',
    'fuel_type': 'Petrol',
    'province': 'Western'
}

X_gam = pd.DataFrame()
X_gam[continuous_cols] = df[continuous_cols]

cat_mappings = {}
for col in categorical_cols:
    categories = [cat_ref[col]] + [c for c in df[col].unique() if c != cat_ref[col]]
    cat_col = pd.Categorical(df[col], categories=categories, ordered=True)
    X_gam[col] = cat_col.codes
    cat_mappings[col] = dict(enumerate(cat_col.categories))

# ------------------------------
# 4. Binary Variables: True=0 reference
# ------------------------------
for col in binary_cols:
    # True = 0 (reference), False = 1
    X_gam[col] = (~df[col].astype(bool)).astype(int)

# ------------------------------
# 5. Construct GAM
# ------------------------------
terms = s(0) + s(1)  # log_engine_cc + age
current_idx = 2

for _ in categorical_cols:
    terms += f(current_idx)
    current_idx += 1

for _ in binary_cols:
    terms += l(current_idx)
    current_idx += 1

X = X_gam.values
y = df['price_transformed'].values  # assume log-price or transformed price

gam = LinearGAM(terms).fit(X, y)

# ------------------------------
# 6. Partial Dependence Plotting
# ------------------------------
feature_names = continuous_cols + categorical_cols + binary_cols
fig, axs = plt.subplots(3, 4, figsize=(24, 15))
axs = axs.flatten()

plot_idx = 0
for term_idx, term in enumerate(gam.terms):
    if term.isintercept:
        continue
    if plot_idx >= len(axs):
        break

    ax = axs[plot_idx]
    f_idx = term.feature
    f_name = feature_names[f_idx]

    # ------------------------------
    # 6a. Discrete Features (categorical / binary)
    # ------------------------------
    if f_name in cat_mappings or f_name in binary_cols:
        # Explicitly include both 0 and 1 for binaries
        if f_name in binary_cols:
            unique_codes = np.array([0, 1])
        else:
            unique_codes = np.sort(X_gam[f_name].unique())

        XX = gam.generate_X_grid(term=term_idx, n=len(unique_codes))
        XX[:, f_idx] = unique_codes
        pdep, confi = gam.partial_dependence(term=term_idx, X=XX, width=0.95)

        # Error bars for discrete points
        ax.errorbar(XX[:, f_idx], pdep,
                    yerr=[pdep - confi[:, 0], confi[:, 1] - pdep],
                    fmt='o', color='tab:red', capsize=5, markersize=8)

    # ------------------------------
    # 6b. Continuous Features
    # ------------------------------
    else:
        XX = gam.generate_X_grid(term=term_idx)
        pdep, confi = gam.partial_dependence(term=term_idx, X=XX, width=0.95)
        ax.plot(XX[:, f_idx], pdep, color='tab:blue', linewidth=2.5)
        ax.fill_between(XX[:, f_idx], confi[:, 0], confi[:, 1], alpha=0.2, color='tab:blue')

    # ------------------------------
    # 6c. Axis Labels
    # ------------------------------
    ax.set_title(f_name.replace('_', ' ').title(), fontsize=14, fontweight='bold')

    if f_name in cat_mappings:
        mapping = cat_mappings[f_name]
        ax.set_xticks(list(mapping.keys()))
        ax.set_xticklabels(list(mapping.values()), rotation=45, ha='right')
    elif f_name in binary_cols:
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['True', 'False'])  # Correct labels for flipped binary

    ax.grid(True, alpha=0.2)
    plot_idx += 1

for j in range(plot_idx, len(axs)):
    axs[j].axis('off')

plt.tight_layout()
plt.show()
